In [ ]:
!git clone https://github.com/yastrebksv/TennisProject.git

In [ ]:
%cd TennisProject
!pip install -r requirements.txt

In [ ]:
!pip install catboost

In [ ]:
%cd /content/TennisProject
!python -c "import torch, torchvision, cv2, numpy, pandas, scipy, catboost; print('all imports OK')"

In [ ]:
!gdown 1XEYZ4myUN7QT-NeBYJI0xteLsvs-ZAOl -O model_ball.pt

In [ ]:
!gdown 1Eo5HDnAQE8y_FbOftKZ8pjiojwuy2BmJ -O model_bounce.cbm

In [ ]:
!gdown 1f-Co64ehgq4uddcQm1aFBDtbnyZhQvgG -O model_court.pt

In [ ]:
!ffmpeg -i /content/new_video3.mp4 -vf scale=1280:720 -c:a copy /content/input_720p.mp4

In [ ]:
%cd /content/TennisProject
!python main.py \
  --path_ball_track_model=model_ball.pt \
  --path_court_model=model_court.pt \
  --path_bounce_model=model_bounce.cbm \
  --path_input_video=/content/input_25fps.mp4 \
  --path_output_video=/content/output.mp4

In [ ]:
!ffmpeg -y -i /content/new_video3.mp4 -c:v libx264 -preset veryfast -r 25 -an /content/input_25fps.mp4

In [ ]:
!ffprobe -v error -select_streams v:0 -show_entries stream=r_frame_rate -of default=noprint_wrappers=1 /content/input_25fps.mp4

In [ ]:
!pip install scenedetect

In [ ]:
!pip install "scenedetect[opencv]==0.5.6.1"

In [ ]:
%cd /content/TennisProject
!python main.py \
  --path_ball_track_model=model_ball.pt \
  --path_court_model=model_court.pt \
  --path_bounce_model=model_bounce.cbm \
  --path_input_video=/content/input_720p.mp4 \
  --path_output_video=/content/output.mp4

In [ ]:
!grep -rl "np.Inf" /content/TennisProject/ | xargs sed -i 's/np\.Inf/np.inf/g'

In [ ]:
from google.colab import files
files.download('/content/output.mp4')

In [ ]:
# Append a save line to main.py right after bounces is computed
!sed -i "/bounces = bounce_detector.predict/a\\    import json; json.dump([int(b) for b in bounces], open('/content/bounces.json','w')); print('Saved bounces:', list(bounces))" /content/TennisProject/main.py

In [ ]:
# === FULL REBUILD: run this first after any session reset ===
%cd /content
!git clone https://github.com/yastrebksv/TennisProject.git
!pip install catboost
!pip install "scenedetect[opencv]==0.5.6.1"
%cd /content/TennisProject
!gdown 1XEYZ4myUN7QT-NeBYJI0xteLsvs-ZAOl -O model_ball.pt
!gdown 1f-Co64ehgq4uddcQm1aFBDtbnyZhQvgG -O model_court.pt
!gdown 1Eo5HDnAQE8y_FbOftKZ8pjiojwuy2BmJ -O model_bounce.cbm
!find /content/TennisProject -name "*.py" -exec sed -i 's/np\.Inf/np.inf/g' {} +
print("REBUILD COMPLETE")

In [ ]:
!pip install -q --upgrade --pre "yt-dlp[default]"

In [ ]:
!yt-dlp \
  -f "bv*[vcodec^=avc1][height<=720]+ba/b" --merge-output-format mp4 \
  --postprocessor-args "ffmpeg:-r 25" \
  "https://www.youtube.com/watch?v=V4BIQoQPAG8" -o "/content/nadal_new1.mp4"

In [ ]:
from scenedetect import VideoManager, SceneManager
from scenedetect.detectors import ContentDetector
import json, os, time, subprocess

assert os.path.ismount('/content/drive'), "Drive not mounted!"
VIDEO='/content/nadal_new1.mp4'
OUTDIR='/content/drive/MyDrive/rallies_nadal_new1'
SEGJSON='/content/drive/MyDrive/segments_nadal_new1.json'

t0=time.time()
vm = VideoManager([VIDEO])
sm = SceneManager()
sm.add_detector(ContentDetector(threshold=27))
vm.set_downscale_factor()
vm.start()
sm.detect_scenes(frame_source=vm)
scenes = sm.get_scene_list(vm.get_base_timecode())
segs=[(s.get_seconds(), e.get_seconds(), e.get_seconds()-s.get_seconds()) for s,e in scenes]
segs=[s for s in segs if 3<=s[2]<=40]
print(f"{len(segs)} candidate rallies in {(time.time()-t0)/60:.0f} min")
vm.release()

json.dump(segs, open(SEGJSON,'w'))
os.makedirs(OUTDIR, exist_ok=True)
for i,(s,e,d) in enumerate(segs):
    out=f'{OUTDIR}/rally_{i:04d}.mp4'
    if os.path.exists(out): continue
    subprocess.run(['ffmpeg','-y','-loglevel','error','-ss',str(s),
                    '-i',VIDEO,'-t',str(d),'-c','copy',out], check=False)
    if i%100==0: print(f"  cut {i}/{len(segs)}")
print("clips ready")

In [ ]:
%cd /content
!git clone https://github.com/yastrebksv/TennisProject.git
!pip install catboost yt-dlp scenedetect -q
%cd /content/TennisProject
!gdown 1XEYZ4myUN7QT-NeBYJI0xteLsvs-ZAOl -O model_ball.pt
!gdown 1f-Co64ehgq4uddcQm1aFBDtbnyZhQvgG -O model_court.pt
!gdown 1Eo5HDnAQE8y_FbOftKZ8pjiojwuy2BmJ -O model_bounce.cbm
!find /content/TennisProject -name "*.py" -exec sed -i 's/np\.Inf/np.inf/g' {} +

from google.colab import drive
drive.mount('/content/drive')

import torch, torchvision, sys
sys.path.append('/content/TennisProject')
from ball_detector import BallDetector
from court_detection_net import CourtDetectorNet
from bounce_detector import BounceDetector

device = 'cuda' if torch.cuda.is_available() else 'cpu'
assert device == 'cuda', "ON CPU — Runtime > Change runtime type > T4 GPU"

models = (
    BallDetector('/content/TennisProject/model_ball.pt', device),
    CourtDetectorNet('/content/TennisProject/model_court.pt', device),
    BounceDetector('/content/TennisProject/model_bounce.cbm'),
    torchvision.models.detection.fasterrcnn_resnet50_fpn(weights='DEFAULT').to(device).eval(),
)
print("Models loaded on", device)

In [ ]:
import cv2, numpy as np, torch, json
from IPython.display import HTML, display
from base64 import b64encode

In [ ]:
import cv2, numpy as np, torch, json
from IPython.display import HTML, display
from base64 import b64encode

def render_tracked(clip_name, models, device, out_path):
    ball_detector, court_detector, bounce_detector, det_model = models
    global H_ref
    path=f'/content/drive/MyDrive/rallies_nadal_new1/{clip_name}'
    cap=cv2.VideoCapture(path); frames=[]
    while True:
        ret,f=cap.read()
        if not ret: break
        frames.append(cv2.resize(f,(1280,720)))
    cap.release()
    H,kps=court_detector.infer_model(frames)
    H_ref = H
    # run the actual tracker on every frame so the video matches what extraction sees
    need=list(range(len(frames)))
    far_t, near_t = track_players_sparse(frames, kps, det_model, device, need)

    w=cv2.VideoWriter(out_path,cv2.VideoWriter_fourcc(*'mp4v'),25,(1280,720))
    for fi in range(len(frames)):
        img=frames[fi].copy()
        if kps[fi] is not None:
            kc=kps[fi]
            def pt(i): return np.array(kc[i]).flatten()[:2]
            cv2.line(img,tuple(pt(0).astype(int)),tuple(pt(1).astype(int)),(255,255,0),2)  # far base
            cv2.line(img,tuple(pt(2).astype(int)),tuple(pt(3).astype(int)),(0,255,255),2)  # near base
        if near_t[fi] is not None:
            b=near_t[fi]
            cv2.rectangle(img,(int(b[0]),int(b[1])),(int(b[2]),int(b[3])),(0,255,0),3)
            cv2.putText(img,"NEAR",(int(b[0]),int(b[1])-8),cv2.FONT_HERSHEY_SIMPLEX,0.7,(0,255,0),2)
        if far_t[fi] is not None:
            b=far_t[fi]
            cv2.rectangle(img,(int(b[0]),int(b[1])),(int(b[2]),int(b[3])),(0,0,255),3)
            cv2.putText(img,"FAR",(int(b[0]),int(b[1])-8),cv2.FONT_HERSHEY_SIMPLEX,0.7,(0,0,255),2)
        else:
            cv2.putText(img,"FAR: none",(20,80),cv2.FONT_HERSHEY_SIMPLEX,0.7,(0,0,255),2)
        cv2.putText(img,f"{clip_name} f{fi}",(20,40),cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,255,255),2)
        w.write(img)
    w.release()
    mp4=open(out_path,'rb').read()
    return "data:video/mp4;base64,"+b64encode(mp4).decode()

real=json.load(open('/content/drive/MyDrive/real_rallies_nadal_new1.json'))
for i,clip in enumerate(real[:2]):
    print(f"rendering {clip}...")
    data=render_tracked(clip, models, device, f'/content/far_{i}.mp4')
    display(HTML(f'<p><b>{clip}</b></p><video width=700 controls><source src="{data}" type="video/mp4"></video>'))

In [ ]:
def render_tracked_clip(clip_name, models, device, out_path='/content/tracked.mp4'):
    ball_detector, court_detector, bounce_detector, det_model = models
    path = f'/content/drive/MyDrive/rallies_nadal_new1/{clip_name}'
    cap=cv2.VideoCapture(path); frames=[]
    while True:
        ret,f=cap.read()
        if not ret: break
        frames.append(cv2.resize(f,(1280,720)))
    cap.release()
    H,kps=court_detector.infer_model(frames)

    # run the ACTUAL tracker so the video matches extraction
    need=list(range(len(frames)))
    far_t, near_t = track_players_sparse(frames, kps, det_model, device, need)

    w=cv2.VideoWriter(out_path,cv2.VideoWriter_fourcc(*'mp4v'),25,(1280,720))
    for fi in range(len(frames)):
        img=frames[fi].copy()
        if kps[fi] is not None:
            quad=build_quad(kps[fi])
            cv2.polylines(img,[quad.astype(int)],True,(255,0,255),2)
        if near_t[fi] is not None:
            b=near_t[fi]
            cv2.rectangle(img,(int(b[0]),int(b[1])),(int(b[2]),int(b[3])),(0,255,0),3)
            cv2.putText(img,"NEAR",(int(b[0]),int(b[1])-8),cv2.FONT_HERSHEY_SIMPLEX,0.7,(0,255,0),2)
        if far_t[fi] is not None:
            b=far_t[fi]
            cv2.rectangle(img,(int(b[0]),int(b[1])),(int(b[2]),int(b[3])),(0,0,255),3)
            cv2.putText(img,"FAR",(int(b[0]),int(b[1])-8),cv2.FONT_HERSHEY_SIMPLEX,0.7,(0,0,255),2)
        else:
            cv2.putText(img,"FAR: none",(20,80),cv2.FONT_HERSHEY_SIMPLEX,0.7,(0,0,255),2)
        cv2.putText(img,f"frame {fi}",(20,40),cv2.FONT_HERSHEY_SIMPLEX,0.8,(255,255,255),2)
        w.write(img)
    w.release()
    print(f"wrote {out_path}")
    mp4=open(out_path,'rb').read()
    data="data:video/mp4;base64,"+b64encode(mp4).decode()
    return HTML(f'<video width=800 controls><source src="{data}" type="video/mp4"></video>')

real=json.load(open('/content/drive/MyDrive/real_rallies_nadal_new1.json'))
for i, clip in enumerate(real[1:5]):   # clips 1-4 (you already saw clip 0)
    print(f"rendering {i+1}/4: {clip}")
    render_tracked_clip(clip, models, device, f'/content/test_{i}.mp4')

In [ ]:
import os, cv2, csv, json, time
import numpy as np
import torch, torchvision

# ---------- COURT GEOMETRY ----------
X_LDBL, X_LSGL, X_CTR, X_RSGL, X_RDBL = 286, 423, 832, 1242, 1379
Y_FAR_BASE, Y_FAR_SVC, Y_NET, Y_NEAR_SVC, Y_NEAR_BASE = 561, 1110, 1748, 2386, 2935
COURT_X, COURT_Y, MARGIN = (286, 1379), (561, 2935), 30
SINGLES_HALF_W = (X_RSGL - X_LSGL) / 2
HALF_LEN       = (Y_NET - Y_FAR_BASE) / 2
COURT_CTR      = (X_CTR, (Y_FAR_BASE + Y_NEAR_BASE) / 2)

FOOT_FRAC = 0.85  # <-- tracking fix: sample player feet above the box bottom.
                   #     confirm via diagnostic; box bottom (1.0) overshoots.

def bounce_region(x, y):
    if y < Y_NET:
        side, deep = 'far', y < (Y_FAR_BASE + Y_NET)/2
    else:
        side, deep = 'near', y > (Y_NET + Y_NEAR_BASE)/2
    band = 'deep' if deep else 'short'
    q = (X_RSGL - X_LSGL)/4
    if   x < X_LSGL + q: lane = 'wide_l'
    elif x < X_CTR:      lane = 'mid_l'
    elif x < X_RSGL - q: lane = 'mid_r'
    else:                lane = 'wide_r'
    return f"{side}_{band}_{lane}"

def zone_of(coord):
    if coord is None: return "unknown"
    x, y = coord
    if not (X_LDBL <= x <= X_RDBL and Y_FAR_BASE <= y <= Y_NEAR_BASE):
        return "out_of_bounds"
    return bounce_region(x, y)

def depth_band(y, side):
    if side == 'far': d = (y - Y_FAR_BASE) / (Y_NET - Y_FAR_BASE)
    else:             d = (Y_NEAR_BASE - y) / (Y_NEAR_BASE - Y_NET)
    d = max(0.0, min(1.0, d))
    if d < 0.20: return 'DEFENSE'
    if d < 0.40: return 'CONTROL'
    if d < 0.60: return 'PRESSURE'
    if d < 0.80: return 'ATTACK'
    return 'FINISH'

def shot_risk(x, y, side):
    d_side = min(x - X_LSGL, X_RSGL - x)
    if side == 'far': d_len = min(y - Y_FAR_BASE, Y_NET - y)
    else:             d_len = min(Y_NEAR_BASE - y, y - Y_NET)
    return round(float(max(1 - d_side/SINGLES_HALF_W, 1 - d_len/HALF_LEN)), 3)

# ---------- POSE ----------
POSE_MODEL = None
def init_pose(device):
    global POSE_MODEL
    if POSE_MODEL is None:
        POSE_MODEL = torchvision.models.detection.keypointrcnn_resnet50_fpn(
            weights='DEFAULT').to(device).eval()
    return POSE_MODEL

def pose_at(frame, box, device, min_score=0.9):
    if box is None: return None
    m = init_pose(device)
    t = torch.from_numpy(frame.transpose(2,0,1)/255).unsqueeze(0).float().to(device)
    with torch.no_grad(): p = m(t)[0]
    if len(p['scores']) == 0: return None
    bx, by = (box[0]+box[2])/2, (box[1]+box[3])/2
    best, bd = None, 1e9
    for kps, sc, bb in zip(p['keypoints'], p['scores'], p['boxes']):
        if float(sc) < min_score: continue
        cx, cy = (float(bb[0])+float(bb[2]))/2, (float(bb[1])+float(bb[3]))/2
        d = np.hypot(cx-bx, cy-by)
        if d < bd: bd, best = d, kps.cpu().numpy()
    return best

def pose_features(kp, handed, prefix):
    out = {}
    if kp is None: return out
    def ok(i): return kp[i,2] > 0.5
    W  = 10 if handed == 'right' else 9
    E  =  8 if handed == 'right' else 7
    SH =  6 if handed == 'right' else 5
    if not (ok(5) and ok(6)): return out
    sw = abs(kp[6,0] - kp[5,0])
    if sw < 5: return out
    out[f'{prefix}_shoulder_w'] = round(float(sw), 1)
    if ok(11) and ok(12):
        out[f'{prefix}_torso_rot'] = round(float(((kp[6,0]-kp[5,0]) -
                                                  (kp[12,0]-kp[11,0])) / sw), 3)
        out[f'{prefix}_lean'] = round(float(((kp[5,0]+kp[6,0])/2 -
                                             (kp[11,0]+kp[12,0])/2) / sw), 3)
        out[f'{prefix}_hip_x'] = round(float((kp[11,0]+kp[12,0])/2), 1)
    if ok(15) and ok(16):
        out[f'{prefix}_stance_w'] = round(float(abs(kp[16,0]-kp[15,0]) / sw), 3)
    if ok(11) and ok(13) and ok(15):
        out[f'{prefix}_knee_flex'] = round(float((kp[13,1]-kp[11,1]) /
                                                 ((kp[15,1]-kp[13,1]) + 1e-6)), 3)
    if ok(W) and ok(12):
        out[f'{prefix}_wrist_h'] = round(float((kp[12,1]-kp[W,1]) / sw), 3)
    if ok(SH) and ok(W):
        out[f'{prefix}_arm_ext'] = round(float(np.hypot(kp[W,0]-kp[SH,0],
                                                        kp[W,1]-kp[SH,1]) / sw), 3)
    if ok(E) and ok(W):
        vx, vy = kp[W,0]-kp[E,0], kp[W,1]-kp[E,1]
        n = np.hypot(vx, vy)
        if n > 1e-6:
            reach = sw * 1.2
            out[f'{prefix}_racket_x'] = round(float(kp[W,0] + vx/n*reach), 1)
            out[f'{prefix}_racket_y'] = round(float(kp[W,1] + vy/n*reach), 1)
    return out

# ---------- BALL / BOUNCE ----------
def interpolate_gaps(ball_track, max_gap=60):
    track = list(ball_track)
    is_real = [t[0] is not None for t in track]
    det = [i for i in range(len(track)) if track[i][0] is not None]
    for k in range(len(det)-1):
        i0, i1 = det[k], det[k+1]
        gap = i1 - i0 - 1
        if 1 <= gap <= max_gap:
            x0,y0 = track[i0]; x1,y1 = track[i1]
            for j in range(1, gap+1):
                t = j/(gap+1)
                track[i0+j] = (x0+(x1-x0)*t, y0+(y1-y0)*t)
    return track, is_real

def to_court(f, ball_track, H):
    if f is None: return None
    pos, mat = ball_track[f], H[f]
    if pos[0] is None or mat is None: return None
    pt = np.array(pos, dtype=np.float32).reshape(1,1,2)
    o = cv2.perspectiveTransform(pt, mat)
    return (float(o[0,0,0]), float(o[0,0,1]))

def player_to_court(box, H_frame, foot_frac=FOOT_FRAC):
    """Feet -> court. Sample ABOVE the box bottom (foot_frac<1): the detection
    box bottom sits below the real feet (shadow/padding) and the near-baseline
    homography amplifies that error into hundreds of court units."""
    if box is None or H_frame is None: return (None, None)
    fx = (box[0] + box[2]) / 2
    fy = box[1] + (box[3] - box[1]) * foot_frac
    foot = np.array([[[fx, fy]]], dtype=np.float32)
    p = cv2.perspectiveTransform(foot, H_frame)[0,0]
    return (round(float(p[0]),1), round(float(p[1]),1))

def on_court(p, margin=MARGIN):
    if p is None: return None
    x, y = p
    return (COURT_X[0]-margin <= x <= COURT_X[1]+margin and
            COURT_Y[0]-margin <= y <= COURT_Y[1]+margin)

def valid_court_pos(p, x_margin=200, y_margin=250):
    """Tightened: players stand behind the baseline but not 6 metres behind.
    Rejects the homography garbage that used to land 400-700 units past."""
    if p is None or p[0] is None or p[1] is None: return False
    x, y = p
    return (COURT_X[0]-x_margin <= x <= COURT_X[1]+x_margin and
            COURT_Y[0]-y_margin <= y <= COURT_Y[1]+y_margin)

def valid_bounce(p, tol=40):
    if p is None or p[0] is None or p[1] is None: return False
    x, y = p
    return (X_LSGL - tol <= x <= X_RSGL + tol and
            Y_FAR_BASE - tol <= y <= Y_NEAR_BASE + tol)

def player_to_court_robust(track, f, H, window=3):
    if f is None: return (None, None)
    for off in [0, -1, 1, -2, 2, -3, 3]:
        g = f + off
        if 0 <= g < len(track) and track[g] is not None and H[g] is not None:
            p = player_to_court(track[g], H[g])
            if valid_court_pos(p):
                return p
    return (None, None)

def dir_change(f, ball_track, w=3):
    ys = [ball_track[i][1] if 0 <= i < len(ball_track) and ball_track[i][0] is not None
          else None for i in range(f-w, f+w+1)]
    before = [y for y in ys[:w]   if y is not None]
    after  = [y for y in ys[w+1:] if y is not None]
    if not before or not after or ys[w] is None: return None
    return ys[w] >= max(before)-2 and ys[w] >= max(after)-2

def classify_bounces(bounces, ball_track, H, cluster_window=8):
    classified = []
    for f in sorted(bounces):
        p = to_court(f, ball_track, H)
        classified.append((f, p, on_court(p), dir_change(f, ball_track)))
    results, i, n = [], 0, len(classified)
    while i < n:
        f, p, oc, dc = classified[i]
        if oc is True:
            j = i
            while (j+1 < n and classified[j+1][2] is True
                   and classified[j+1][0]-classified[j][0] <= cluster_window):
                j += 1
            results.append((classified[j][0], classified[j][1], 'IN'))
            i = j + 1
        elif oc is False:
            had = any(r[2]=='IN' for r in results)
            results.append((f, p, 'FALSE' if (had and dc) else 'OUT'))
            i += 1
        else:
            i += 1
    collapsed, run, prev = [], [], None
    for e in results:
        if e[2] != 'IN':
            if run: collapsed.append(run[-1]); run = []
            prev = None; collapsed.append(e); continue
        s = 'far' if e[1][1] < Y_NET else 'near'
        # only merge into the current run if SAME side AND close in time (true duplicate);
        # a same-side bounce far apart in time is a separate real bounce
        if s == prev and run and (e[0] - run[-1][0]) <= cluster_window:
            run.append(e)
        else:
            if run: collapsed.append(run[-1])
            run = [e]; prev = s
    if run: collapsed.append(run[-1])
    return collapsed

def track_players_sparse(frames, kps_court, det_model, device, need_frames):
    def detect(img, min_score=0.85):
        t = torch.from_numpy(img.transpose(2,0,1)/255).unsqueeze(0).float().to(device)
        with torch.no_grad(): pred = det_model(t)[0]
        return [b.cpu().numpy() for b,l,s in zip(pred['boxes'],pred['labels'],pred['scores'])
                if l.item()==1 and s.item()>min_score]
    def foot(b): return np.array([(b[0]+b[2])/2, b[3]])

    far_t  = [None]*len(frames)
    near_t = [None]*len(frames)
    far_pos = near_pos = None       # last known good foot positions (persist through gaps)
    MAX_MATCH = 200                 # px; nearest detection farther than this = not that player

    for f in sorted(set(x for x in need_frames if x is not None and 0<=x<len(frames))):
        kc = kps_court[f]
        if kc is None: continue
        quad = build_quad(kc)
        players = [b for b in detect(frames[f])
                   if 40 < (b[3]-b[1]) < 420 and touches_quad(b, quad)]
        if not players: continue

        if far_pos is None or near_pos is None:
            # seed identity by frame height (far = highest in frame, near = lowest)
            players.sort(key=lambda b: b[3])
            far_box  = players[0]
            near_box = players[-1] if len(players) > 1 else None
        else:
            # continuity: match each player to nearest detection to their last position,
            # but only if it's within MAX_MATCH (else the player is lost this frame -> None)
            far_cand  = min(players, key=lambda b: np.linalg.norm(foot(b)-far_pos))
            near_cand = min(players, key=lambda b: np.linalg.norm(foot(b)-near_pos))
            far_box  = far_cand  if np.linalg.norm(foot(far_cand)-far_pos)   < MAX_MATCH else None
            near_box = near_cand if np.linalg.norm(foot(near_cand)-near_pos) < MAX_MATCH else None
            # collision: same box matched to both -> give to closer, other None
            if far_box is not None and far_box is near_box:
                if np.linalg.norm(foot(far_box)-far_pos) < np.linalg.norm(foot(near_box)-near_pos):
                    near_box = None
                else:
                    far_box = None

        far_t[f], near_t[f] = far_box, near_box
        # update last-known ONLY on a valid match — a miss keeps the prior position
        if far_box  is not None: far_pos  = foot(far_box)
        if near_box is not None: near_pos = foot(near_box)

    return far_t, near_t

    def is_stationary(b):
        fp = np.array(foot(b))
        return any(np.linalg.norm(fp - s) < STAT_RADIUS*1.4 for s in stationary_spots)

    # PASS 3: per frame, keep only MOVING people (players); far=top, near=bottom
    for f in idxs:
        ppl = per_frame.get(f, [])
        players = [b for b in ppl if not is_stationary(b)]
        if not players: continue
        players.sort(key=lambda b: b[3])       # by foot image-y: smallest=highest in frame
        far_t[f]  = players[0]                  # highest in frame = far player
        near_t[f] = players[-1]                 # lowest in frame = near player
    return far_t, near_t

def find_contact(bf, ball_track, track, look_ahead=25, thresh=120):
    best = None
    for f in range(bf, min(bf+look_ahead, len(ball_track))):
        b, box = ball_track[f], track[f]
        if b[0] is None or box is None: continue
        cx, cy = (box[0]+box[2])/2, (box[1]+box[3])/2
        d = np.hypot(b[0]-cx, b[1]-cy)
        if best is None or d < best[1]: best = (f, d)
    if best is None or best[1] > thresh: return None
    return best[0]

def dist(a, b):
    if a is None or b is None or None in a or None in b: return None
    return round(float(np.hypot(a[0]-b[0], a[1]-b[1])), 1)

# global handle so track_players_sparse can gate on the homography
H_ref = None

def extract_shots_v3(clip_path, rally_id, models, device,
                     near_handed='right', far_handed='right',
                     near_name='PLAYER', far_name='PLAYER',
                     surface='HARD', look_ahead=25, contact_thresh=120):
    global H_ref
    ball_detector, court_detector, bounce_detector, det_model = models
    cap = cv2.VideoCapture(clip_path)
    frames = []
    while True:
        ret, f = cap.read()
        if not ret: break
        frames.append(cv2.resize(f, (1280,720)))
    cap.release()
    if len(frames) < 30: return []

    ball_track   = interpolate_gaps(ball_detector.infer_model(frames), max_gap=60)
    H, kps_court = court_detector.infer_model(frames)
    H_ref = H
    bounces      = bounce_detector.predict([b[0] for b in ball_track],
                                           [b[1] for b in ball_track])
    classified   = classify_bounces(bounces, ball_track, H)

    in_b = [(bf, c) for bf, c, v in classified if v == 'IN']
    if len(in_b) < 2: return []

    prelim = [f for bf,_ in in_b
                for f in range(bf, min(bf+look_ahead, len(frames)))]
    far_p, near_p = track_players_sparse(frames, kps_court, det_model, device, prelim)
    contacts = {}
    for bf, recv in in_b:
        side  = 'far' if recv[1] < Y_NET else 'near'
        track = far_p if side == 'far' else near_p
        contacts[bf] = find_contact(bf, ball_track, track, look_ahead, contact_thresh)

    need = set()
    for bf, _ in in_b:
        for off in range(-3, 4):
            need.add(bf + off)
        cf = contacts[bf]
        if cf is not None:
            for off in range(-3, 4):
                need.add(cf + off)
    far_t, near_t = track_players_sparse(frames, kps_court, det_model, device, need)

    rows = []
    for i in range(len(in_b)-1):
        bf,  recv = in_b[i]
        nbf, tgt  = in_b[i+1]
        side     = 'far' if recv[1] < Y_NET else 'near'
        tgt_side = 'far' if tgt[1]  < Y_NET else 'near'
        if tgt_side == side: continue
        if not valid_court_pos(recv) or not valid_court_pos(tgt): continue
        if not valid_bounce(recv): continue

        recv_trk = far_t  if side=='far' else near_t
        hit_trk  = near_t if side=='far' else far_t
        recv_handed = far_handed if side=='far' else near_handed
        hit_handed  = near_handed if side=='far' else far_handed
        recv_name   = far_name   if side=='far' else near_name

        cf  = contacts[bf]
        pbf = in_b[i-1][0] if i > 0 else None
        t0  = contacts[pbf] if pbf is not None else None

        if cf is None:
            rows.append(dict(
                rally_id=rally_id, shot_num=i+1, is_serve=(i==0),
                player_side=side, player_name=recv_name, handedness=recv_handed,
                surface=surface, t0_hit_contact=t0, t1_bounce=bf,
                t2_recv_contact=None, t3_next_bounce=nbf,
                bounce_x=round(recv[0],1), bounce_y=round(recv[1],1),
                bounce_region=zone_of(recv), recv_depth=depth_band(recv[1], side),
                stroke='NO_CONTACT', outcome='WINNER', is_winner=True))
            break

        rx1, ry1 = player_to_court_robust(recv_trk, bf, H)
        hx1, hy1 = player_to_court_robust(hit_trk,  bf, H)
        rx2, ry2 = player_to_court_robust(recv_trk, cf, H)
        hx2, hy2 = player_to_court_robust(hit_trk,  cf, H)
        rx0, ry0 = player_to_court_robust(recv_trk, t0, H)   # receiver at hitter's contact (t0)
        if not valid_court_pos((rx1,ry1)): rx1 = ry1 = None
        if not valid_court_pos((hx1,hy1)): hx1 = hy1 = None
        if not valid_court_pos((rx2,ry2)): rx2 = ry2 = None
        if not valid_court_pos((hx2,hy2)): hx2 = hy2 = None
        if not valid_court_pos((rx0,ry0)): rx0 = ry0 = None
        # DISPLACEMENT: receiver's travel from hitter's contact (t0) to own contact (t2)
        displacement = dist((rx0,ry0), (rx2,ry2))
        ball_c   = to_court(cf, ball_track, H)

        t_flight = (bf - t0) if t0 is not None else None
        t_prep   = cf - bf
        t_total  = (cf - t0) if t0 is not None else None

        prev_c = to_court(t0, ball_track, H)
        ball_speed = None
        if prev_c and t_flight and t_flight > 0:
            fd = dist(prev_c, recv)
            if fd: ball_speed = round(fd/t_flight, 2)

        recv_travelled = dist((rx1,ry1), (rx2,ry2))
        recv_speed = round(recv_travelled/t_prep,2) if recv_travelled and t_prep>0 else None
        hit_recov  = dist((hx1,hy1), (hx2,hy2))
        hit_speed  = round(hit_recov/t_prep,2) if hit_recov and t_prep>0 else None
        hit_ctr    = dist((hx2,hy2), COURT_CTR)

        P = {}
        P.update(pose_features(pose_at(frames[bf], recv_trk[bf], device), recv_handed, 'r1'))
        P.update(pose_features(pose_at(frames[cf], recv_trk[cf], device), recv_handed, 'r2'))
        P.update(pose_features(pose_at(frames[bf], hit_trk[bf],  device), hit_handed,  'h1'))
        P.update(pose_features(pose_at(frames[cf], hit_trk[cf],  device), hit_handed,  'h2'))
        if t0 is not None:
            P.update(pose_features(pose_at(frames[t0], recv_trk[t0], device),
                                   recv_handed, 'r0'))

        for k in ['torso_rot','knee_flex','stance_w','lean','arm_ext','wrist_h']:
            a, b = P.get(f'r1_{k}'), P.get(f'r2_{k}')
            if a is not None and b is not None:
                P[f'r_d_{k}'] = round(b-a, 2)

        hip = P.get('r2_hip_x')
        if hip is not None and ball_track[cf][0] is not None:
            P['ball_ahead_of_hips'] = round(float(ball_track[cf][0]-hip), 1)

        if recv_trk[cf] is not None and ball_track[cf][0] is not None:
            d_x = ball_track[cf][0]
            c_x = (recv_trk[cf][0]+recv_trk[cf][2])/2
            screen = 'right' if d_x > c_x else 'left'
            body   = screen if side=='near' else ('left' if screen=='right' else 'right')
            stroke = 'FOREHAND' if body == recv_handed else 'BACKHAND'
        else:
            stroke = 'UNKNOWN'
        same   = (rx2 is not None) and ((rx2 < X_CTR) == (tgt[0] < X_CTR))

        row = dict(
            rally_id=rally_id, shot_num=i+1, is_serve=(i==0),
            player_side=side, player_name=recv_name, handedness=recv_handed,
            surface=surface,
            t0_hit_contact=t0, t1_bounce=bf, t2_recv_contact=cf, t3_next_bounce=nbf,
            t_flight=t_flight, t_prep=t_prep, t_total=t_total,
            bounce_x=round(recv[0],1), bounce_y=round(recv[1],1),
            bounce_region=zone_of(recv), recv_depth=depth_band(recv[1], side),
            recv_x=rx1, recv_y=ry1, hitter_x=hx1, hitter_y=hy1,
            recv_contact_x=rx2, recv_contact_y=ry2,
            hitter_at_t2_x=hx2, hitter_at_t2_y=hy2,
            contact_ball_x=round(ball_c[0],1) if ball_c else None,
            contact_ball_y=round(ball_c[1],1) if ball_c else None,
            ball_speed=ball_speed,
            recv_travelled=recv_travelled, recv_closing_speed=recv_speed,
            hitter_recovery_dist=hit_recov, hitter_recovery_speed=hit_speed,
            hitter_dist_from_centre=hit_ctr,
            recv_at_t0_x=rx0, recv_at_t0_y=ry0, displacement=displacement,
            stroke=stroke,
            direction='DOWN_THE_LINE' if same else 'CROSSCOURT',
            tgt_x=round(tgt[0],1), tgt_y=round(tgt[1],1),
            tgt_region=zone_of(tgt), risk=shot_risk(tgt[0], tgt[1], tgt_side),
            outcome='NEUTRAL', is_winner=False)
        row.update(P)
        if not valid_bounce(tgt):
            row['outcome'] = 'ERROR'
            row['tgt_x'] = row['tgt_y'] = row['tgt_region'] = None
            row['risk'] = row['direction'] = None
            rows.append(row)
            continue
        rows.append(row)

    for i, r in enumerate(rows):
        r['prev_displacement'] = rows[i-1].get('displacement') if i > 0 else None

    if rows and rows[-1].get('outcome') == 'NEUTRAL':
        rows[-1]['outcome'] = 'UNKNOWN'
    return rows

BASE_FIELDS = [
    'rally_id','shot_num','is_serve','player_side','player_name','handedness','surface',
    't0_hit_contact','t1_bounce','t2_recv_contact','t3_next_bounce',
    't_flight','t_prep','t_total',
    'bounce_x','bounce_y','bounce_region','recv_depth',
    'recv_x','recv_y','hitter_x','hitter_y',
    'recv_contact_x','recv_contact_y','hitter_at_t2_x','hitter_at_t2_y',
    'contact_ball_x','contact_ball_y',
    'ball_speed','recv_travelled','recv_closing_speed',
    'hitter_recovery_dist','hitter_recovery_speed','hitter_dist_from_centre',
    'recv_at_t0_x','recv_at_t0_y',
    'stroke','direction','tgt_x','tgt_y','tgt_region','risk',
    'outcome','is_winner','displacement','prev_displacement']
POSE_KEYS = ['torso_rot','lean','stance_w','knee_flex','wrist_h','arm_ext',
             'racket_x','racket_y','hip_x']
FIELDS = (BASE_FIELDS
          + [f'{p}_{k}' for p in ['r0','r1','r2','h1','h2'] for k in POSE_KEYS]
          + [f'r_d_{k}' for k in ['torso_rot','knee_flex','stance_w','lean','arm_ext','wrist_h']]
          + ['ball_ahead_of_hips'])
print(f"v3 extractor defined — {len(FIELDS)} fields, FOOT_FRAC={FOOT_FRAC}")

In [ ]:
def build_quad(kc, sidek=1.08, far_ext=0.12, near_ext=0.15):
    def pt(i):
        return np.array(kc[i], float).flatten()[:2]
    fL, fR, nL, nR = pt(0), pt(1), pt(2), pt(3)
    cx = (fL[0] + fR[0] + nL[0] + nR[0]) / 4
    court_h = ((nL[1] + nR[1]) / 2) - ((fL[1] + fR[1]) / 2)
    def corner(p, is_far):
        x = cx + (p[0] - cx) * sidek
        y = p[1] + (-court_h * far_ext if is_far else court_h * near_ext)
        return [x, y]
    return np.array([corner(fL, True), corner(fR, True),
                     corner(nR, False), corner(nL, False)], dtype=np.float32)

def touches_quad(b, quad):
    pts = [((b[0]+b[2])/2, b[3]), (b[0], b[3]), (b[2], b[3]),
           (b[0], b[1]), (b[2], b[1]), ((b[0]+b[2])/2, (b[1]+b[3])/2)]
    return any(cv2.pointPolygonTest(quad, (float(px), float(py)), False) >= 0
               for px, py in pts)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json, numpy as np
real = json.load(open('/content/drive/MyDrive/real_rallies_nadal_new1.json'))

for clip in real[:5]:
    rid = clip.replace('.mp4','')
    rows = extract_shots_v3(f'/content/drive/MyDrive/rallies_nadal_new1/{clip}',
                            rid, models, device, surface='HARD')
    print(f"\n=== {clip}: {len(rows)} shots ===")
    for r in rows:
        bx = r.get('bounce_x')
        off = abs(bx-832) if bx is not None else None
        print(f"  shot {r.get('shot_num')}: side={r.get('player_side')}, "
              f"bounce_x={bx}, off_center={off}, "
              f"displacement={r.get('displacement')}, "
              f"recv_at_t0=({r.get('recv_at_t0_x')},{r.get('recv_at_t0_y')}), "
              f"recv_contact=({r.get('recv_contact_x')},{r.get('recv_contact_y')})")

In [ ]:
import json
real = json.load(open('/content/drive/MyDrive/real_rallies_nadal_new1.json'))
rows = []
for clip in real[:20]:
    rid = clip.replace('.mp4','')
    rows += extract_shots_v3(f'/content/drive/MyDrive/rallies_nadal_new1/{clip}',
                             rid, models, device, surface='HARD')

# how often is displacement None, and WHY?
total = len(rows)
first_shots = sum(1 for r in rows if r.get('shot_num')==1)  # no t0, correct None
has_disp = sum(1 for r in rows if r.get('displacement') is not None)
t0_none_nonfirst = sum(1 for r in rows if r.get('shot_num')!=1
                       and r.get('recv_at_t0_x') is None)
t2_none = sum(1 for r in rows if r.get('recv_contact_x') is None)

print(f"{total} shots")
print(f"  with displacement:        {has_disp} ({100*has_disp/total:.0f}%)")
print(f"  first shots (correct None): {first_shots}")
print(f"  non-first w/ t0 lost:      {t0_none_nonfirst}")
print(f"  t2 (own contact) lost:     {t2_none}")

In [ ]:
import numpy as np
# does displacement track how far the shot pushed the receiver?
# compare displacement for shots landing WIDE vs MIDDLE
wide, mid = [], []
for r in test_rows:
    d = r.get('displacement')
    bx = r.get('bounce_x')   # where the ball came to the receiver
    if d is None or bx is None: continue
    # how far from center was the ball placed?
    off_center = abs(bx - 832)   # X_CTR = 832
    if off_center > 300: wide.append(d)
    else: mid.append(d)

print(f"WIDE shots (ball >300 from center): n={len(wide)}, median displacement {np.median(wide):.0f}" if wide else "no wide")
print(f"MIDDLE shots (ball <300 from center): n={len(mid)}, median displacement {np.median(mid):.0f}" if mid else "no middle")

In [ ]:
!pip install -q --upgrade "scenedetect[opencv]>=0.6.4"

In [ ]:
%cd /content/TennisProject
!python main.py \
  --path_ball_track_model=model_ball.pt \
  --path_court_model=model_court.pt \
  --path_bounce_model=model_bounce.cbm \
  --path_input_video=/content/input3_720p.mp4 \
  --path_output_video=/content/output.mp4

In [ ]:
import json, numpy as np

# temporarily loosen the margins
def valid_court_pos(p, x_margin=280, y_margin=350):   # loosened from 200, 250
    if p is None or p[0] is None or p[1] is None: return False
    x, y = p
    return (COURT_X[0]-x_margin <= x <= COURT_X[1]+x_margin and
            COURT_Y[0]-y_margin <= y <= COURT_Y[1]+y_margin)

real = json.load(open('/content/drive/MyDrive/real_rallies_nadal_new1.json'))
rows = []
for clip in real[:8]:
    rid = clip.replace('.mp4','')
    rows += extract_shots_v3(f'/content/drive/MyDrive/rallies_nadal_new1/{clip}',
                             rid, models, device, surface='HARD')

# coverage
total = len(rows)
has_disp = sum(1 for r in rows if r.get('displacement') is not None)
print(f"{total} shots, {has_disp} with displacement ({100*has_disp/total:.0f}%)")

# are the RECOVERED positions sensible? check how far past the baseline they land
far_pos = [(r['recv_contact_x'], r['recv_contact_y']) for r in rows
           if r.get('player_side')=='far' and r.get('recv_contact_y') is not None]
near_pos = [(r['recv_contact_x'], r['recv_contact_y']) for r in rows
            if r.get('player_side')=='near' and r.get('recv_contact_y') is not None]
print(f"\nfar-side contact y (should be near {Y_FAR_BASE}=561):")
if far_pos:
    ys=[p[1] for p in far_pos]
    print(f"  median {np.median(ys):.0f}, range {min(ys):.0f}-{max(ys):.0f}")
    print(f"  past far baseline (y<561): {sum(1 for y in ys if y<561)}/{len(ys)}, "
          f"worst overshoot: {561-min(ys):.0f}")
print(f"near-side contact y (should be near {Y_NEAR_BASE}=2935):")
if near_pos:
    ys=[p[1] for p in near_pos]
    print(f"  median {np.median(ys):.0f}, range {min(ys):.0f}-{max(ys):.0f}")

# displacement values still sensible?
disps=[r['displacement'] for r in rows if r.get('displacement') is not None]
if disps:
    d=np.array(disps)
    print(f"\ndisplacement: median {np.median(d):.0f}, IQR {np.percentile(d,25):.0f}-{np.percentile(d,75):.0f}, "
          f"max {d.max():.0f}, %over1500 {100*np.mean(d>1500):.0f}%")

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import cv2, numpy as np, torch, json
from IPython.display import HTML, display
from base64 import b64encode

def render_tracked(clip_name, models, device, out_path):
    ball_detector, court_detector, bounce_detector, det_model = models
    path=f'/content/drive/MyDrive/rallies_nadal_new1/{clip_name}'
    cap=cv2.VideoCapture(path); frames=[]
    while True:
        ret,f=cap.read()
        if not ret: break
        frames.append(cv2.resize(f,(1280,720)))
    cap.release()
    H,kps=court_detector.infer_model(frames)
    def d2line(p,a,b):
        a,b,p=np.array(a,float),np.array(b,float),np.array(p,float)
        if np.allclose(a,b): return np.linalg.norm(p-a)
        return abs(float(np.cross(b-a,p-a)))/np.linalg.norm(b-a)
    w=cv2.VideoWriter(out_path,cv2.VideoWriter_fourcc(*'mp4v'),25,(1280,720))
    for fi in range(len(frames)):
        img=frames[fi].copy(); kc=kps[fi]
        if kc is not None:
            def pt(i): return np.array(kc[i]).flatten()[:2]
            fa,fb,na,nb=pt(0),pt(1),pt(2),pt(3)
            t=torch.from_numpy(img.transpose(2,0,1)/255).unsqueeze(0).float().to(device)
            with torch.no_grad(): pred=det_model(t)[0]
            boxes=[b.cpu().numpy() for b,l,s in zip(pred['boxes'],pred['labels'],pred['scores'])
                   if l.item()==1 and s.item()>0.85]
            fbest,nbest,farbox,nearbox=1e9,1e9,None,None
            for b in boxes:
                h=b[3]-b[1]
                if h<40 or h>420: continue
                foot=((b[0]+b[2])/2,b[3])
                df,dn=d2line(foot,fa,fb),d2line(foot,na,nb)
                if df<fbest: fbest,farbox=df,b
                if dn<nbest: nbest,nearbox=dn,b
            if nearbox is not None:
                cv2.rectangle(img,(int(nearbox[0]),int(nearbox[1])),(int(nearbox[2]),int(nearbox[3])),(0,255,0),3)
                cv2.putText(img,"NEAR",(int(nearbox[0]),int(nearbox[1])-8),cv2.FONT_HERSHEY_SIMPLEX,0.7,(0,255,0),2)
            if farbox is not None:
                cv2.rectangle(img,(int(farbox[0]),int(farbox[1])),(int(farbox[2]),int(farbox[3])),(0,0,255),3)
                cv2.putText(img,"FAR",(int(farbox[0]),int(farbox[1])-8),cv2.FONT_HERSHEY_SIMPLEX,0.7,(0,0,255),2)
        cv2.putText(img,f"{clip_name} f{fi}",(20,40),cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,255,255),2)
        w.write(img)
    w.release()
    mp4=open(out_path,'rb').read()
    data="data:video/mp4;base64,"+b64encode(mp4).decode()
    return data

real=json.load(open('/content/drive/MyDrive/real_rallies_nadal_new1.json'))

# render 12 clips and show them all
for i, clip in enumerate(real[:12]):
    print(f"rendering {i+1}/12: {clip}")
    data=render_tracked(clip, models, device, f'/content/tracked_{i}.mp4')
    display(HTML(f'<p><b>{clip}</b></p><video width=700 controls><source src="{data}" type="video/mp4"></video>'))

In [ ]:
import cv2, numpy as np, torch, json
from IPython.display import HTML, display
from base64 import b64encode

def render_tracked(clip_name, models, device, out_path):
    ball_detector, court_detector, bounce_detector, det_model = models
    global H_ref
    path=f'/content/drive/MyDrive/rallies_nadal_new1/{clip_name}'
    cap=cv2.VideoCapture(path); frames=[]
    while True:
        ret,f=cap.read()
        if not ret: break
        frames.append(cv2.resize(f,(1280,720)))
    cap.release()
    H,kps=court_detector.infer_model(frames)
    H_ref=H
    need=list(range(len(frames)))
    far_t, near_t = track_players_sparse(frames, kps, det_model, device, need)

    w=cv2.VideoWriter(out_path,cv2.VideoWriter_fourcc(*'mp4v'),25,(1280,720))
    for fi in range(len(frames)):
        img=frames[fi].copy()
        if kps[fi] is not None:
            kc=kps[fi]
            def pt(i): return np.array(kc[i]).flatten()[:2]
            cv2.line(img,tuple(pt(0).astype(int)),tuple(pt(1).astype(int)),(255,255,0),2)
            cv2.line(img,tuple(pt(2).astype(int)),tuple(pt(3).astype(int)),(0,255,255),2)
        if near_t[fi] is not None:
            b=near_t[fi]
            cv2.rectangle(img,(int(b[0]),int(b[1])),(int(b[2]),int(b[3])),(0,255,0),3)
            cv2.putText(img,"NEAR",(int(b[0]),int(b[1])-8),cv2.FONT_HERSHEY_SIMPLEX,0.7,(0,255,0),2)
        if far_t[fi] is not None:
            b=far_t[fi]
            cv2.rectangle(img,(int(b[0]),int(b[1])),(int(b[2]),int(b[3])),(0,0,255),3)
            cv2.putText(img,"FAR",(int(b[0]),int(b[1])-8),cv2.FONT_HERSHEY_SIMPLEX,0.7,(0,0,255),2)
        else:
            cv2.putText(img,"FAR: none",(20,80),cv2.FONT_HERSHEY_SIMPLEX,0.7,(0,0,255),2)
        cv2.putText(img,f"{clip_name} f{fi}",(20,40),cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,255,255),2)
        w.write(img)
    w.release()
    mp4=open(out_path,'rb').read()
    return "data:video/mp4;base64,"+b64encode(mp4).decode()

real=json.load(open('/content/drive/MyDrive/real_rallies_nadal_new1.json'))
for i,clip in enumerate(real[:2]):
    print(f"rendering {clip}...")
    data=render_tracked(clip, models, device, f'/content/stat_{i}.mp4')
    display(HTML(f'<p><b>{clip}</b></p><video width=700 controls><source src="{data}" type="video/mp4"></video>'))

In [ ]:
def render_combined(clip_name, models, device, out_path='/content/combined.mp4',
                    near_handed='left', look_ahead=25):
    ball_detector, court_detector, bounce_detector, det_model = models
    cap=cv2.VideoCapture(f'/content/drive/MyDrive/rallies_nadal_new1/{clip_name}'); frames=[]
    while True:
        ret,f=cap.read()
        if not ret: break
        frames.append(cv2.resize(f,(1280,720)))
    cap.release()
    ball_track, ball_is_real = interpolate_gaps(ball_detector.infer_model(frames), max_gap=60)
    H, kps_court = court_detector.infer_model(frames)
    bounces = bounce_detector.predict([b[0] for b in ball_track],[b[1] for b in ball_track])
    classified = classify_bounces(bounces, ball_track, H)
    in_b = [(bf,c) for bf,c,v in classified if v=='IN']
    in_bounce_frames = {bf for bf,_ in in_b}
    all_bounce_frames = set(bounces)
    far_t, near_t = track_players_sparse(frames, kps_court, det_model, device, need_frames=range(len(frames)))

    # near arm_ext every frame
    arm=[None]*len(frames)
    for fi in range(len(frames)):
        kp = match_pose(pose_frame(frames[fi], device), near_t[fi])
        if kp is not None:
            arm[fi] = pose_features(kp, near_handed, 'x').get('x_arm_ext')

    # near-player contact bands (peak->trough of arm_ext) per near-side bounce
    band_frames=set()
    for bf, recv in in_b:
        if recv[1] < Y_NET: continue
        pts=[(f,arm[f]) for f in range(bf,min(bf+look_ahead,len(frames))) if arm[f] is not None]
        if len(pts)<3: continue
        best=None
        for i in range(1,len(pts)-1):
            if not (pts[i-1][1] < pts[i][1] > pts[i+1][1]): continue
            j=i
            while j+1<len(pts) and pts[j+1][1] < pts[j][1]: j+=1
            if j>i:
                drop=pts[i][1]-pts[j][1]
                if best is None or drop>best[2]: best=(pts[i][0],pts[j][0],drop)
        if best:
            for f in range(best[0],best[1]+1): band_frames.add(f)

    w=cv2.VideoWriter(out_path,cv2.VideoWriter_fourcc(*'mp4v'),25,(1280,720))
    for fi in range(len(frames)):
        img=frames[fi].copy()

        # --- ball: current position + short trail ---
        for k in range(max(0,fi-6), fi+1):
            b=ball_track[k]
            if b[0] is not None:
                shade=int(255*(k-(fi-6))/6)
                cv2.circle(img,(int(b[0]),int(b[1])),3,(shade,shade,0),-1)
        if ball_track[fi][0] is not None:
            cv2.circle(img,(int(ball_track[fi][0]),int(ball_track[fi][1])),6,(255,255,0),2)

        # --- bounces: flash a marker on bounce frames ---
        if fi in in_bounce_frames:
            cv2.putText(img,"BOUNCE (IN)",(500,120),cv2.FONT_HERSHEY_SIMPLEX,0.9,(0,140,255),3)
        elif fi in all_bounce_frames:
            cv2.putText(img,"bounce",(520,120),cv2.FONT_HERSHEY_SIMPLEX,0.7,(120,120,120),2)

        # --- near player box + contact band ---
        b=near_t[fi]
        if b is not None:
            inb = fi in band_frames
            cv2.rectangle(img,(int(b[0]),int(b[1])),(int(b[2]),int(b[3])),
                          (0,255,255) if inb else (0,255,0), 4 if inb else 2)
            if inb:
                cv2.putText(img,"** CONTACT **",(int(b[0]),int(b[1])-10),
                            cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,255,255),2)
        # --- far player box (no contact system yet) ---
        bf_=far_t[fi]
        if bf_ is not None:
            cv2.rectangle(img,(int(bf_[0]),int(bf_[1])),(int(bf_[2]),int(bf_[3])),(0,0,255),2)

        a=arm[fi]
        cv2.putText(img,f"near arm_ext:{a:.2f}" if a else "near arm_ext:--",
                    (20,70),cv2.FONT_HERSHEY_SIMPLEX,0.7,(0,255,0),2)
        cv2.putText(img,f"frame {fi}",(20,40),cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,255,255),2)
        w.write(img)
    w.release()
    mp4=open(out_path,'rb').read()
    return HTML(f'<video width=800 controls><source src="{"data:video/mp4;base64,"+b64encode(mp4).decode()}"></video>')

In [ ]:
render_combined('rally_0032.mp4', models, device,
                out_path='/content/drive/MyDrive/combined_0017.mp4')

In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/combined_0017.mp4')

In [ ]:
def match_pose(dets, box):
    if box is None or not dets: return None
    bx, by = (box[0]+box[2])/2, (box[1]+box[3])/2
    best, bd = None, 1e9
    for kp, cx, cy in dets:
        d = np.hypot(cx-bx, cy-by)
        if d < bd: bd, best = d, kp
    return best

def pose_frame(frame, device, min_score=0.9):
    m = init_pose(device)
    t = torch.from_numpy(frame.transpose(2,0,1)/255).unsqueeze(0).float().to(device)
    with torch.no_grad(): p = m(t)[0]
    return [(kp.cpu().numpy(), (float(bb[0])+float(bb[2]))/2, (float(bb[1])+float(bb[3]))/2)
            for kp, sc, bb in zip(p['keypoints'], p['scores'], p['boxes']) if float(sc) >= min_score]

In [ ]:
import inspect
print(inspect.getsource(interpolate_gaps))

In [ ]:
import cv2
cap = cv2.VideoCapture('/content/input3_720p.mp4')
print("fps:", cap.get(cv2.CAP_PROP_FPS), "frames:", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
cap.release()

In [ ]:
import sys
sys.path.append('/content/TennisProject')
import cv2, torch
from ball_detector import BallDetector
from court_detection_net import CourtDetectorNet
from bounce_detector import BounceDetector

device = 'cuda' if torch.cuda.is_available() else 'cpu'
cap = cv2.VideoCapture('/content/input3_720p.mp4')   # <-- use the Melbourne file's actual name here
frames = []
while True:
    ret, f = cap.read()
    if not ret: break
    frames.append(f)
cap.release()
print(f"Loaded {len(frames)} frames")

ball_detector = BallDetector('/content/TennisProject/model_ball.pt', device)
ball_track = ball_detector.infer_model(frames)
court_detector = CourtDetectorNet('/content/TennisProject/model_court.pt', device)
homography_matrices, kps_court = court_detector.infer_model(frames)
bounce_detector = BounceDetector('/content/TennisProject/model_bounce.cbm')
bounces = bounce_detector.predict([x[0] for x in ball_track], [x[1] for x in ball_track])
print("DONE. Bounces:", sorted(bounces))

In [ ]:
import numpy as np

def interpolate_small_gaps(ball_track, max_gap=4):
    """Fill only short gaps (<= max_gap consecutive None frames) by linear interpolation.
    Long gaps are left as None to avoid fabricating trajectory."""
    track = list(ball_track)
    n = len(track)

    # find indices where ball is detected
    detected = [i for i in range(n) if track[i][0] is not None]
    if not detected:
        return track

    filled = 0
    for k in range(len(detected) - 1):
        i0 = detected[k]
        i1 = detected[k + 1]
        gap = i1 - i0 - 1
        if 1 <= gap <= max_gap:
            x0, y0 = track[i0]
            x1, y1 = track[i1]
            for j in range(1, gap + 1):
                t = j / (gap + 1)
                xi = x0 + (x1 - x0) * t
                yi = y0 + (y1 - y0) * t
                track[i0 + j] = (xi, yi)
                filled += 1
    print(f"Filled {filled} frames in gaps of <= {max_gap}. Long gaps left untouched.")
    return track

# Apply to the live ball_track
ball_track_interp = interpolate_small_gaps(ball_track, max_gap=4)

# Re-run bounce detection on the interpolated track
x_ball = [x[0] for x in ball_track_interp]
y_ball = [x[1] for x in ball_track_interp]
bounces_interp = bounce_detector.predict(x_ball, y_ball)
print("Bounces BEFORE interpolation:", sorted(bounces))
print("Bounces AFTER  interpolation:", sorted(bounces_interp))

In [ ]:
import cv2
import numpy as np

# Re-read frames if not in memory (they should still be there from detection)
out_path = '/content/output_interp.mp4'
h, w = frames[0].shape[:2]
fps = 25  # this clip is 25fps
out = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

bounce_set = set(bounces_interp)
trace = 7  # how many past frames of ball trail to draw

for i in range(len(frames)):
    img = frames[i].copy()

    # draw ball trail (interpolated track)
    for j in range(trace):
        idx = i - j
        if idx >= 0 and ball_track_interp[idx][0] is not None:
            x = int(ball_track_interp[idx][0])
            y = int(ball_track_interp[idx][1])
            # fade older points
            color = (0, 255, 0)
            cv2.circle(img, (x, y), max(1, 4 - j//2), color, -1)

    # mark bounces: when current frame is a bounce, draw a big red ring at ball pos
    if i in bounce_set and ball_track_interp[i][0] is not None:
        x = int(ball_track_interp[i][0])
        y = int(ball_track_interp[i][1])
        cv2.circle(img, (x, y), 12, (0, 0, 255), 3)
        cv2.putText(img, "BOUNCE", (x + 14, y), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)

    # frame counter
    cv2.putText(img, f"Frame {i} ({i/fps:.1f}s)", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)

    out.write(img)

out.release()
print("Wrote", out_path)

In [ ]:
# Big interpolation test - fills gaps up to 60 frames with straight lines.
# WARNING: long gaps get straight-line fills that may not match the ball's real path.
ball_track_big = interpolate_small_gaps(ball_track, max_gap=60)

x_ball = [x[0] for x in ball_track_big]
y_ball = [x[1] for x in ball_track_big]
bounces_big = bounce_detector.predict(x_ball, y_ball)

print("Bounces with light interp (max_gap=4):", sorted(bounces_interp))
print("Bounces with BIG interp  (max_gap=60):", sorted(bounces_big))

In [ ]:
import cv2

out_path = '/content/output_big_interp.mp4'
h, w = frames[0].shape[:2]
fps = 25
out = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
bounce_set = set(bounces_big)
trace = 7

for i in range(len(frames)):
    img = frames[i].copy()
    for j in range(trace):
        idx = i - j
        if idx >= 0 and ball_track_big[idx][0] is not None:
            x = int(ball_track_big[idx][0]); y = int(ball_track_big[idx][1])
            cv2.circle(img, (x, y), max(1, 4 - j//2), (0, 255, 0), -1)
    if i in bounce_set and ball_track_big[i][0] is not None:
        x = int(ball_track_big[i][0]); y = int(ball_track_big[i][1])
        cv2.circle(img, (x, y), 12, (0, 0, 255), 3)
        cv2.putText(img, "BOUNCE", (x + 14, y), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)
    cv2.putText(img, f"Frame {i} ({i/fps:.1f}s)", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)
    out.write(img)
out.release()
print("Wrote", out_path)

In [ ]:
from google.colab import files
files.download('/content/output_big_interp.mp4')

In [ ]:
def find_y_peaks(ball_track, min_prominence=15):
    """Find frames where ball reaches a local y-maximum (lowest on screen)
    while moving down before and up after - candidate bounces by physics."""
    peaks = []
    track = ball_track
    n = len(track)
    for i in range(2, n - 2):
        # need valid points around i
        window = [track[j] for j in range(i-2, i+3)]
        if any(p[0] is None for p in window):
            continue
        ys = [p[1] for p in window]  # y at i-2, i-1, i, i+1, i+2
        # i is a local max in y (lowest point), ball came down then went up
        if ys[2] == max(ys) and ys[2] - min(ys[0], ys[4]) >= min_prominence:
            peaks.append(i)
    return peaks

y_peak_candidates = find_y_peaks(ball_track_big, min_prominence=15)
print("Model bounces:      ", sorted(bounces_big))
print("Y-peak candidates:  ", y_peak_candidates)

In [ ]:
model_set = set(bounces_big)
peak_set = set(y_peak_candidates)

agreed = sorted(model_set & peak_set)      # both methods agree - high confidence
model_only = sorted(model_set - peak_set)  # model inferred (often gap-hidden bounces)
peak_only = sorted(peak_set - model_set)   # physics found (possible misses or racket hits)

print("High confidence (both agree):", agreed)
print("Model only (check: gap bounces?):", model_only)
print("Peak only (check: real bounce or racket hit?):", peak_only)

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Build y-position over time, using interpolated track
ys = np.array([p[1] if p[0] is not None else np.nan for p in ball_track_big], dtype=float)
xs = np.array([p[0] if p[0] is not None else np.nan for p in ball_track_big], dtype=float)

# First derivative: vertical velocity (change in y per frame)
vy = np.gradient(ys)
# Second derivative: vertical acceleration
ay = np.gradient(vy)

# Speed (magnitude of motion) for reference
vx = np.gradient(xs)
speed = np.sqrt(vx**2 + vy**2)

frames = np.arange(len(ys))

fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)
axes[0].plot(frames, ys); axes[0].set_ylabel('y (position)'); axes[0].invert_yaxis()
axes[0].set_title('Ball vertical position (inverted: down = bounce)')
axes[1].plot(frames, vy, color='orange'); axes[1].axhline(0, color='k', lw=0.5)
axes[1].set_ylabel('dy/dt (vert velocity)'); axes[1].set_title('Vertical velocity - zero crossings = direction reversals')
axes[2].plot(frames, ay, color='red'); axes[2].set_ylabel('accel'); axes[2].set_title('Vertical acceleration - spikes = bounces/hits')
axes[3].plot(frames, speed, color='green'); axes[3].set_ylabel('speed'); axes[3].set_title('Overall speed')

# mark model bounces on all charts
for b in bounces_big:
    for ax in axes:
        ax.axvline(b, color='blue', ls='--', alpha=0.4)

axes[3].set_xlabel('frame')
plt.tight_layout()
plt.savefig('/content/derivative_chart.png', dpi=90)
print("Saved chart.")

In [ ]:
from google.colab import files
files.download('/content/derivative_chart.png')

In [ ]:
import numpy as np
import cv2

# Court boundary in reference coords (from your vars(ref)).
# Doubles: x 286-1379. Singles: x 423-1242. y baseline-to-baseline: 561-2935.
COURT_X_MIN, COURT_X_MAX = 286, 1379   # <-- doubles; change to 423,1242 for singles
COURT_Y_MIN, COURT_Y_MAX = 561, 2935
MARGIN = 30  # pixels of tolerance near lines

def to_court(frame, ball_track, homography_matrices):
    """Transform a ball position to reference-court coords. Returns (x,y) or None."""
    pos = ball_track[frame]
    mat = homography_matrices[frame]
    if pos[0] is None or mat is None:
        return None
    pt = np.array(pos, dtype=np.float32).reshape(1,1,2)
    out = cv2.perspectiveTransform(pt, mat)
    return (out[0,0,0], out[0,0,1])

def on_court(cx, cy, margin=MARGIN):
    return (COURT_X_MIN - margin <= cx <= COURT_X_MAX + margin and
            COURT_Y_MIN - margin <= cy <= COURT_Y_MAX + margin)

def has_direction_change(frame, ball_track, window=3):
    """Vertical velocity reverses sign around this frame (down then up)."""
    ys = []
    for i in range(frame-window, frame+window+1):
        if 0 <= i < len(ball_track) and ball_track[i][0] is not None:
            ys.append(ball_track[i][1])
        else:
            ys.append(None)
    # need before and after
    before = [y for y in ys[:window] if y is not None]
    after  = [y for y in ys[window+1:] if y is not None]
    if not before or not after or ys[window] is None:
        return None  # can't tell
    # bounce: ball was descending (y increasing) then ascending (y decreasing)
    descending_before = ys[window] >= max(before) - 2
    ascending_after   = ys[window] >= max(after) - 2
    return descending_before and ascending_after

def filter_bounces(bounces, ball_track, homography_matrices, cluster_window=8):
    results = []
    for f in sorted(bounces):
        court = to_court(f, ball_track, homography_matrices)
        dir_change = has_direction_change(f, ball_track)
        if court is None:
            results.append((f, None, 'no_homography', 'KEEP (unverifiable)'))
            continue
        cx, cy = court
        inside = on_court(cx, cy)
        if inside:
            label = 'IN'
        else:
            # outside the court
            if dir_change:
                label = 'FALSE (out + direction change)'   # Rule 3
            else:
                label = 'OUT'                               # Rule 2
        results.append((f, (round(cx), round(cy)), 'in' if inside else 'out', label))
    return results

filtered = filter_bounces(bounces_big, ball_track_big, homography_matrices)
print("Frame | court-coord | zone | verdict")
for f, coord, zone, verdict in filtered:
    print(f"  {f:3d} | {coord} | {zone} | {verdict}")

In [ ]:
import numpy as np, cv2

COURT_X_MIN, COURT_X_MAX = 286, 1379   # doubles; use 423,1242 for singles
COURT_Y_MIN, COURT_Y_MAX = 561, 2935
MARGIN = 30

def _to_court(f, ball_track, H):
    pos = ball_track[f]; mat = H[f]
    if pos[0] is None or mat is None: return None
    pt = np.array(pos, dtype=np.float32).reshape(1,1,2)
    o = cv2.perspectiveTransform(pt, mat)
    return (float(o[0,0,0]), float(o[0,0,1]))

def _on_court(p, margin=MARGIN):
    if p is None: return None
    x, y = p
    return (COURT_X_MIN-margin <= x <= COURT_X_MAX+margin and
            COURT_Y_MIN-margin <= y <= COURT_Y_MAX+margin)

def _dir_change(f, ball_track, w=3):
    ys = [ball_track[i][1] if 0<=i<len(ball_track) and ball_track[i][0] is not None else None
          for i in range(f-w, f+w+1)]
    before = [y for y in ys[:w] if y is not None]
    after  = [y for y in ys[w+1:] if y is not None]
    if not before or not after or ys[w] is None: return None
    return ys[w] >= max(before)-2 and ys[w] >= max(after)-2

def analyze_bounces(bounces, ball_track, H, cluster_window=8):
    """Rule 1 (cluster->latest on-court), then Rules 2/3 for outside bounces,
    using shot context (whether a real on-court bounce already happened)."""
    bset = sorted(bounces)
    # First: classify each bounce's court position
    classified = []
    for f in bset:
        p = _to_court(f, ball_track, H)
        classified.append((f, p, _on_court(p), _dir_change(f, ball_track)))

    # Rule 1: cluster consecutive on-court bounces within cluster_window,
    # keep the LATEST (furthest in travel) as the true bounce.
    results = []
    i = 0
    n = len(classified)
    while i < n:
        f, p, on, dc = classified[i]
        if on is True:
            # gather cluster of on-court bounces close in time
            j = i
            while j+1 < n and classified[j+1][2] is True and classified[j+1][0]-classified[j][0] <= cluster_window:
                j += 1
            # latest in cluster = true bounce
            tf, tp, _, _ = classified[j]
            results.append((tf, tp, 'IN (true bounce)'))
            i = j + 1
        elif on is False:
            # outside: Rule 3 if a real bounce already happened AND direction change; else Rule 2
            had_real = any(r[2].startswith('IN') for r in results)
            if had_real and dc:
                results.append((f, p, 'FALSE (out+dir-change)'))
            elif not dc:
                results.append((f, p, 'OUT'))
            else:
                results.append((f, p, 'FALSE (out+dir-change)'))
            i += 1
        else:
            results.append((f, None, 'KEEP (no homography)'))
            i += 1
    return results

analysis = analyze_bounces(bounces_big, ball_track_big, homography_matrices)
print("Frame | court-coord | verdict")
for f, coord, verdict in analysis:
    c = (round(coord[0]), round(coord[1])) if coord else None
    print(f"  {f:3d} | {c} | {verdict}")

In [ ]:
import numpy as np, cv2

# ---- Reference court geometry ----
X_LDBL, X_LSGL, X_CTR, X_RSGL, X_RDBL = 286, 423, 832, 1242, 1379
Y_FAR_BASE, Y_FAR_SVC, Y_NET, Y_NEAR_SVC, Y_NEAR_BASE = 561, 1110, 1748, 2386, 2935
COURT_X_MIN, COURT_X_MAX = 286, 1379
COURT_Y_MIN, COURT_Y_MAX = 561, 2935
MARGIN = 30

ZONES = {
    "far_alley_left":     (X_LDBL, X_LSGL, Y_FAR_BASE, Y_NET),
    "far_alley_right":    (X_RSGL, X_RDBL, Y_FAR_BASE, Y_NET),
    "near_alley_left":    (X_LDBL, X_LSGL, Y_NET, Y_NEAR_BASE),
    "near_alley_right":   (X_RSGL, X_RDBL, Y_NET, Y_NEAR_BASE),
    "far_service_left":   (X_LSGL, X_CTR,  Y_FAR_SVC, Y_NET),
    "far_service_right":  (X_CTR,  X_RSGL, Y_FAR_SVC, Y_NET),
    "far_backcourt_left": (X_LSGL, X_CTR,  Y_FAR_BASE, Y_FAR_SVC),
    "far_backcourt_right":(X_CTR,  X_RSGL, Y_FAR_BASE, Y_FAR_SVC),
    "near_service_left":  (X_LSGL, X_CTR,  Y_NET, Y_NEAR_SVC),
    "near_service_right": (X_CTR,  X_RSGL, Y_NET, Y_NEAR_SVC),
    "near_backcourt_left":(X_LSGL, X_CTR,  Y_NEAR_SVC, Y_NEAR_BASE),
    "near_backcourt_right":(X_CTR, X_RSGL, Y_NEAR_SVC, Y_NEAR_BASE),
}

def _to_court(f, ball_track, H):
    pos, mat = ball_track[f], H[f]
    if pos[0] is None or mat is None: return None
    pt = np.array(pos, dtype=np.float32).reshape(1,1,2)
    o = cv2.perspectiveTransform(pt, mat)
    return (float(o[0,0,0]), float(o[0,0,1]))

def _on_court(p, margin=MARGIN):
    if p is None: return None
    x, y = p
    return (COURT_X_MIN-margin <= x <= COURT_X_MAX+margin and
            COURT_Y_MIN-margin <= y <= COURT_Y_MAX+margin)

def _dir_change(f, ball_track, w=3):
    ys = [ball_track[i][1] if 0<=i<len(ball_track) and ball_track[i][0] is not None else None
          for i in range(f-w, f+w+1)]
    before = [y for y in ys[:w] if y is not None]
    after  = [y for y in ys[w+1:] if y is not None]
    if not before or not after or ys[w] is None: return None
    return ys[w] >= max(before)-2 and ys[w] >= max(after)-2

def _zone(cx, cy):
    for name,(x0,x1,y0,y1) in ZONES.items():
        if x0 <= cx <= x1 and y0 <= cy <= y1:
            return name
    return "out_of_bounds"

def _side(cy):
    return "far" if cy < Y_NET else "near"

def analyze_bounces(bounces, ball_track, H, cluster_window=8):
    bset = sorted(bounces)
    classified = [(f, _to_court(f,ball_track,H), _on_court(_to_court(f,ball_track,H)), _dir_change(f,ball_track)) for f in bset]

    results = []
    i, n = 0, len(classified)
    while i < n:
        f, p, on, dc = classified[i]
        if on is True:
            j = i
            while j+1 < n and classified[j+1][2] is True and classified[j+1][0]-classified[j][0] <= cluster_window:
                j += 1
            tf, tp, _, _ = classified[j]
            results.append((tf, tp, 'IN (true bounce)'))
            i = j + 1
        elif on is False:
            had_real = any(r[2].startswith('IN') for r in results)
            if had_real and dc:
                results.append((f, p, 'FALSE (out+dir-change)'))
            elif not dc:
                results.append((f, p, 'OUT'))
            else:
                results.append((f, p, 'FALSE (out+dir-change)'))
            i += 1
        else:
            results.append((f, None, 'KEEP (no homography)'))
            i += 1
    return results

analysis = analyze_bounces(bounces_big, ball_track_big, homography_matrices)
print(f"{'Frame':>5} | {'court-coord':>13} | {'side':>4} | {'zone':<20} | verdict")
print("-"*75)
for f, coord, verdict in analysis:
    if coord:
        cx, cy = coord
        print(f"{f:>5} | ({round(cx):4d},{round(cy):5d}) | {_side(cy):>4} | {_zone(cx,cy):<20} | {verdict}")
    else:
        print(f"{f:>5} | {'(none)':>13} | {'--':>4} | {'--':<20} | {verdict}")

In [ ]:
import numpy as np, cv2

def draw_zones(analysis, out_path='/content/zones_view.png', scale=0.35):
    W, H = int(1665*scale), int(3506*scale)
    img = np.full((H, W, 3), 30, np.uint8)
    def sx(x): return int(x*scale)
    def sy(y): return int(y*scale)

    zone_colors = {
        "far_alley_left":(80,80,120),  "far_alley_right":(80,80,120),
        "near_alley_left":(80,120,80), "near_alley_right":(80,120,80),
        "far_service_left":(120,90,60),  "far_service_right":(150,110,70),
        "far_backcourt_left":(90,70,110),"far_backcourt_right":(110,80,130),
        "near_service_left":(60,120,90), "near_service_right":(70,140,110),
        "near_backcourt_left":(70,110,90),"near_backcourt_right":(80,130,100),
    }
    for name,(x0,x1,y0,y1) in ZONES.items():
        c = zone_colors.get(name,(70,70,70))
        cv2.rectangle(img,(sx(x0),sy(y0)),(sx(x1),sy(y1)),c,-1)
        cv2.rectangle(img,(sx(x0),sy(y0)),(sx(x1),sy(y1)),(200,200,200),1)
        # zone label
        cv2.putText(img,name.replace('_','\n'),(sx(x0)+3,sy(y0)+14),
                    cv2.FONT_HERSHEY_SIMPLEX,0.28,(220,220,220),1)

    cv2.line(img,(sx(286),sy(1748)),(sx(1379),sy(1748)),(0,0,255),2)  # net

    for f, coord, verdict in analysis:
        if coord is None: continue
        cx, cy = coord
        px, py = sx(cx), sy(cy)
        if 'IN' in verdict:   color=(0,255,0)
        elif verdict=='OUT':  color=(0,140,255)
        else:                 color=(0,0,255)
        cv2.circle(img,(px,py),5,color,-1)
        cv2.circle(img,(px,py),5,(255,255,255),1)
        cv2.putText(img,str(f),(px+7,py+4),cv2.FONT_HERSHEY_SIMPLEX,0.4,(255,255,255),1)

    cv2.imwrite(out_path, img)
    print("Saved", out_path)

draw_zones(analysis)

In [ ]:
from google.colab import files
files.download('/content/zones_view.png')

In [ ]:
def collapse_by_side(analysis):
    """Collapse consecutive same-side IN bounces into one shot,
    keeping the LAST (furthest along travel direction) as the true bounce.
    A side flip marks a new shot."""
    # keep only real IN bounces with a known side for this pass
    kept = []
    run = []  # current same-side run
    def side_of(entry):
        coord = entry[1]
        if coord is None: return None
        return 'far' if coord[1] < 1748 else 'near'

    def flush(run):
        if run:
            # last in run = furthest along travel = true bounce
            kept.append(run[-1])

    prev_side = None
    for entry in analysis:
        f, coord, verdict = entry
        if not verdict.startswith('IN'):
            # non-IN (OUT/FALSE/no-homography): flush current run, keep as-is separately
            flush(run); run = []
            prev_side = None
            kept.append(entry)  # preserve OUT/FALSE markers
            continue
        s = side_of(entry)
        if s == prev_side:
            run.append(entry)          # same side: extend current shot
        else:
            flush(run)                  # side flipped: close previous shot
            run = [entry]
            prev_side = s
    flush(run)
    return kept

collapsed = collapse_by_side(analysis)
print("Frame |   court-coord | side | zone | verdict")
for f, coord, verdict in collapsed:
    if coord:
        cx, cy = coord
        side = 'far' if cy < 1748 else 'near'
        # reuse your zone function if defined; else omit
        print(f"  {f:3d} | ({round(cx):4d},{round(cy):5d}) | {side:4s} | {verdict}")
    else:
        print(f"  {f:3d} | (none) | -- | {verdict}")

In [ ]:
def ball_near_box(b, box, pad=25):
    """Is the ball within `pad` pixels of the player's box edges?"""
    if box is None or b[0] is None:
        return False
    x1, y1, x2, y2 = box
    return (x1-pad <= b[0] <= x2+pad) and (y1-pad <= b[1] <= y2+pad)

def detect_shots(track, ball_track, which, handed='right', pad=25):
    shots = []
    armed = True          # ready to register a new shot
    run = []              # frames where ball is currently near the box

    for f in range(len(ball_track)):
        b = ball_track[f]
        box = track[f]
        near = ball_near_box(b, box, pad)

        if near and armed:
            run.append((f, b[0], (box[0]+box[2])/2))   # (frame, ball_x, box_center_x)
        elif not near and run:
            # ball just left the box -> register ONE shot from the run, then reset
            # pick the frame where ball_x is closest to box center (contact moment)
            cf, bx, cx = min(run, key=lambda r: abs(r[1]-r[2]))
            screen = 'right' if bx > cx else 'left'
            body = screen if which=='near' else ('left' if screen=='right' else 'right')
            shots.append((cf, 'FOREHAND' if body==handed else 'BACKHAND'))
            run = []
            armed = False          # disarm until ball is clearly away
        elif not near:
            armed = True           # ball is away from box -> re-arm for next shot

    # flush any trailing run
    if run:
        cf, bx, cx = min(run, key=lambda r: abs(r[1]-r[2]))
        screen = 'right' if bx > cx else 'left'
        body = screen if which=='near' else ('left' if screen=='right' else 'right')
        shots.append((cf, 'FOREHAND' if body==handed else 'BACKHAND'))
    return shots

near_shots = detect_shots(near_track, ball_track_big, 'near', handed='right', pad=25)
far_shots  = detect_shots(far_track,  ball_track_big, 'far',  handed='right', pad=25)
print("NEAR shots:", near_shots)
print("FAR shots:", far_shots)

In [ ]:
def detect_shots_after_bounce(track, ball_track, analysis, which, handed='right', pad=25):
    # in-court bounce frames, sorted
    in_bounces = sorted(f for f, coord, verdict in analysis if verdict.startswith('IN'))

    def had_in_bounce_before(frame):
        # was there an in-court bounce before this shot frame?
        return any(b < frame for b in in_bounces)

    def last_in_bounce_before(frame):
        prev = [b for b in in_bounces if b < frame]
        return max(prev) if prev else None

    shots = []
    armed = True
    run = []
    for f in range(len(ball_track)):
        b = ball_track[f]; box = track[f]
        near = ball_near_box(b, box, pad)
        if near and armed:
            run.append((f, b[0], (box[0]+box[2])/2))
        elif not near and run:
            cf, bx, cx = min(run, key=lambda r: abs(r[1]-r[2]))
            # RULE: only register if an in-court bounce preceded this shot
            if had_in_bounce_before(cf):
                screen = 'right' if bx > cx else 'left'
                body = screen if which=='near' else ('left' if screen=='right' else 'right')
                shots.append((cf, 'FOREHAND' if body==handed else 'BACKHAND',
                              last_in_bounce_before(cf)))
            run = []; armed = False
        elif not near:
            armed = True
    return shots

near_shots = detect_shots_after_bounce(near_track, ball_track_big, analysis, 'near', 'right')
far_shots  = detect_shots_after_bounce(far_track,  ball_track_big, analysis, 'far',  'right')
print("NEAR shots (frame, stroke, preceding_bounce):")
for s in near_shots: print("  ", s)
print("FAR shots (frame, stroke, preceding_bounce):")
for s in far_shots: print("  ", s)

In [ ]:
def raw_near_runs(track, ball_track, pad=25):
    runs = []; in_run=False; start=None
    for f in range(len(ball_track)):
        near = ball_near_box(ball_track[f], track[f], pad)
        if near and not in_run: start=f; in_run=True
        elif not near and in_run: runs.append((start,f-1)); in_run=False
    if in_run: runs.append((start,len(ball_track)-1))
    return runs

for which, track in [('near',near_track),('far',far_track)]:
    for pad in [25, 40, 60]:
        runs = raw_near_runs(track, ball_track_big, pad)
        print(f"{which} pad={pad}: {len(runs)} runs -> {runs}")

In [ ]:
import cv2

out_path = '/content/tracked_players.mp4'
h, w = frames[0].shape[:2]
fps_out = 25  # match your clip's fps
out = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps_out, (w, h))

for f in range(len(frames)):
    img = frames[f].copy()
    for box, color, lbl in [(far_track[f], (255,0,0), 'FAR'),
                            (near_track[f], (0,0,255), 'NEAR')]:
        if box is not None:
            x1,y1,x2,y2 = [int(v) for v in box]
            cv2.rectangle(img, (x1,y1), (x2,y2), color, 2)
            cv2.putText(img, lbl, (x1, y1-6), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    # also draw the ball if present
    b = ball_track_big[f]
    if b[0] is not None:
        cv2.circle(img, (int(b[0]), int(b[1])), 6, (0,255,255), 2)
    # frame counter
    cv2.putText(img, f"F{f}", (10,25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
    out.write(img)

out.release()
print("Wrote", out_path)

In [ ]:
from google.colab import files
files.download('/content/tracked_players.mp4')

In [ ]:
def identify_and_track_players(frames, homography_matrices, gate=120, start_frame=0):
    def center(b): return np.array([(b[0]+b[2])/2, (b[1]+b[3])/2])
    def foot_court(b, inv):
        fs = np.array([[[(b[0]+b[2])/2, b[3]]]], dtype=np.float32)
        return cv2.perspectiveTransform(fs, inv)[0,0]

    FAR_BASELINE_Y = 561
    NET_Y = 1748   # far side is y < NET_Y in court coords

    # first usable frame
    sf = start_frame
    while sf < len(frames):
        if homography_matrices[sf] is not None and len(detect_persons(frames[sf])) >= 2:
            break
        sf += 1
    if sf >= len(frames):
        return [None]*len(frames), [None]*len(frames), None

    inv0 = homography_matrices[sf]
    cands = []
    for b in detect_persons(frames[sf]):
        cx, cy = foot_court(b, inv0)
        cands.append((b, cx, cy))
    # far player = closest to far baseline, among people on the far side
    far_side = [c for c in cands if c[2] < NET_Y]
    near_side = [c for c in cands if c[2] >= NET_Y]
    far0  = min(far_side,  key=lambda c: abs(c[2]-FAR_BASELINE_Y))[0] if far_side else None
    near0 = max(near_side, key=lambda c: c[2])[0] if near_side else None
    last_far, last_near = far0, near0

    far_track, near_track = [None]*len(frames), [None]*len(frames)
    for f in range(sf, len(frames)):
        inv = homography_matrices[f]
        ppl = detect_persons(frames[f])

        # ---- FAR PLAYER: only accept candidates on the far side of the court ----
        if ppl and inv is not None and last_far is not None:
            far_candidates = []
            for b in ppl:
                cx, cy = foot_court(b, inv)
                if cy < NET_Y:                      # must be on far side
                    far_candidates.append(b)
            if far_candidates:
                cents = [center(b) for b in far_candidates]
                i = int(np.argmin([distance.euclidean(center(last_far), c) for c in cents]))
                if distance.euclidean(center(last_far), cents[i]) < gate:
                    last_far = far_candidates[i]
            # if no far-side candidate, hold last_far (don't switch to a near-side person)

        # ---- NEAR PLAYER: standard nearest-box ----
        if ppl and last_near is not None:
            cents = [center(b) for b in ppl]
            i = int(np.argmin([distance.euclidean(center(last_near), c) for c in cents]))
            if distance.euclidean(center(last_near), cents[i]) < gate:
                last_near = ppl[i]

        far_track[f], near_track[f] = last_far, last_near
    return far_track, near_track, sf

far_track, near_track, start_f = identify_and_track_players(frames, homography_matrices)
print(f"Players identified at frame {start_f}")

In [ ]:
import inspect
src = inspect.getsource(identify_and_track_players)
# check the per-frame far-side filter is present
print("has per-frame far filter:", "far_candidates" in src)
print("has cy < NET_Y inside loop:", src.count("NET_Y") >= 2)  # once in setup, once in loop

In [ ]:
def track_locked_players(frames, homography_matrices, far0, near0, gate=80):
    """Lock onto two starting boxes and follow ONLY those two, frame to frame.
    All other detections are ignored. Tight gate prevents jumping to other people."""
    def center(b): return np.array([(b[0]+b[2])/2, (b[1]+b[3])/2])

    far_track  = [None]*len(frames)
    near_track = [None]*len(frames)
    last_far, last_near = far0, near0

    for f in range(len(frames)):
        ppl = detect_persons(frames[f])
        if not ppl:
            far_track[f], near_track[f] = last_far, last_near
            continue
        cents = [center(b) for b in ppl]

        # FAR: only update to a detection within `gate` px of last far position
        if last_far is not None:
            d = [np.linalg.norm(center(last_far)-c) for c in cents]
            i = int(np.argmin(d))
            if d[i] < gate:
                last_far = ppl[i]
            # else: hold last position, do NOT jump

        # NEAR: same, strict follow
        if last_near is not None:
            d = [np.linalg.norm(center(last_near)-c) for c in cents]
            i = int(np.argmin(d))
            if d[i] < gate:
                last_near = ppl[i]

        far_track[f], near_track[f] = last_far, last_near
    return far_track, near_track

# Use the CORRECT starting boxes from frame 0 (or start_f)
far0  = far_track[start_f]    # the correct far box you confirmed at the start
near0 = near_track[start_f]   # the correct near box
far_track, near_track = track_locked_players(frames, homography_matrices, far0, near0, gate=80)
print("Re-tracked with locked players.")

In [ ]:
import cv2
out_path = '/content/tracked_players.mp4'
h, w = frames[0].shape[:2]
out = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), 25, (w, h))
for f in range(len(frames)):
    img = frames[f].copy()
    for box, color, lbl in [(far_track[f],(255,0,0),'FAR'), (near_track[f],(0,0,255),'NEAR')]:
        if box is not None:
            x1,y1,x2,y2 = [int(v) for v in box]
            cv2.rectangle(img,(x1,y1),(x2,y2),color,2)
            cv2.putText(img,lbl,(x1,y1-6),cv2.FONT_HERSHEY_SIMPLEX,0.6,color,2)
    b = ball_track_big[f]
    if b[0] is not None:
        cv2.circle(img,(int(b[0]),int(b[1])),6,(0,255,255),2)
    cv2.putText(img,f"F{f}",(10,25),cv2.FONT_HERSHEY_SIMPLEX,0.7,(0,255,0),2)
    out.write(img)
out.release()
print("Wrote", out_path)
from google.colab import files
files.download('/content/tracked_players.mp4')

In [ ]:
f = 60
kc = kps_court[f]
print("type:", type(kc))
print("first few points:", kc[:4] if kc is not None else "None")

In [ ]:
import numpy as np

def dist_point_to_line(p, a, b):
    a, b, p = np.array(a,dtype=float), np.array(b,dtype=float), np.array(p,dtype=float)
    if np.allclose(a, b): return np.linalg.norm(p-a)
    return abs(np.cross(b-a, p-a)) / np.linalg.norm(b-a)

def get_baseline_pts(kc, which='far'):
    def pt(i): return np.array(kc[i]).flatten()[:2]
    return (pt(0), pt(1)) if which=='far' else (pt(2), pt(3))

def assign_players_by_baseline(frame_idx):
    kc = kps_court[frame_idx]
    if kc is None: return None, None
    far_a, far_b = get_baseline_pts(kc, 'far')
    near_a, near_b = get_baseline_pts(kc, 'near')
    people = detect_persons(frames[frame_idx])
    if not people: return None, None
    far_box, near_box = None, None
    far_best, near_best = 1e9, 1e9
    for b in people:
        foot = ((b[0]+b[2])/2, b[3])
        df = dist_point_to_line(foot, far_a, far_b)
        dn = dist_point_to_line(foot, near_a, near_b)
        if df < far_best:  far_best, far_box = df, b
        if dn < near_best: near_best, near_box = dn, b
    return far_box, near_box

far_track  = [None]*len(frames)
near_track = [None]*len(frames)
for f in range(len(frames)):
    far_track[f], near_track[f] = assign_players_by_baseline(f)
print("Assigned players per-frame by baseline distance.")

In [ ]:
import cv2
out_path = '/content/tracked_players.mp4'
h, w = frames[0].shape[:2]
out = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), 25, (w, h))
for f in range(len(frames)):
    img = frames[f].copy()
    for box, color, lbl in [(far_track[f],(255,0,0),'FAR'), (near_track[f],(0,0,255),'NEAR')]:
        if box is not None:
            x1,y1,x2,y2 = [int(v) for v in box]
            cv2.rectangle(img,(x1,y1),(x2,y2),color,2)
            cv2.putText(img,lbl,(x1,y1-6),cv2.FONT_HERSHEY_SIMPLEX,0.6,color,2)
    b = ball_track_big[f]
    if b[0] is not None:
        cv2.circle(img,(int(b[0]),int(b[1])),6,(0,255,255),2)
    cv2.putText(img,f"F{f}",(10,25),cv2.FONT_HERSHEY_SIMPLEX,0.7,(0,255,0),2)
    out.write(img)
out.release()
print("Wrote", out_path)

In [ ]:
from google.colab import files
files.download('/content/tracked_players.mp4')

In [ ]:
def shots_from_bounces(analysis, ball_track, near_track, far_track,
                       near_handed='right', far_handed='right', fps=25, look_ahead=25):
    in_bounces = [(f, coord) for f, coord, v in analysis if v.startswith('IN')]
    shots = []
    for bf, coord in in_bounces:
        side = 'far' if coord[1] < 1748 else 'near'
        track = far_track if side=='far' else near_track
        handed = far_handed if side=='far' else near_handed
        best = None
        for f in range(bf, min(bf+look_ahead, len(ball_track))):
            b = ball_track[f]; box = track[f]
            if b[0] is None or box is None: continue
            cx = (box[0]+box[2])/2; cy=(box[1]+box[3])/2
            d = np.hypot(b[0]-cx, b[1]-cy)
            if best is None or d < best[1]:
                best = (f, d, b[0], cx)
        if best is None: continue
        cf, d, bx, cx = best
        screen = 'right' if bx > cx else 'left'
        body = screen if side=='near' else ('left' if screen=='right' else 'right')
        stroke = 'FOREHAND' if body==handed else 'BACKHAND'
        shots.append((bf, cf, side, stroke, round(cf/fps,1)))
    return shots

shots = shots_from_bounces(analysis, ball_track_big, near_track, far_track)
print("bounce | contact | player | stroke | time")
print("-"*45)
for bf, cf, side, stroke, t in shots:
    print(f"  {bf:3d}  ->  {cf:3d} | {side:4s} | {stroke:8s} | {t}s")
print(f"\nTotal shots: {len(shots)}")

In [ ]:
shots = shots_from_bounces(analysis, ball_track_big, near_track, far_track,
                           near_handed='left',    # near player LEFT-handed
                           far_handed='right')     # far player RIGHT-handed
print("bounce | contact | player | stroke | time")
print("-"*45)
for bf, cf, side, stroke, t in shots:
    print(f"  {bf:3d}  ->  {cf:3d} | {side:4s} | {stroke:8s} | {t}s")
print(f"\nTotal shots: {len(shots)}")

In [ ]:
import numpy as np

# --- zone lookup (reuses your ZONES dict) ---
def zone_of(coord):
    if coord is None: return "unknown"
    cx, cy = coord
    for name,(x0,x1,y0,y1) in ZONES.items():
        if x0 <= cx <= x1 and y0 <= cy <= y1:
            return name
    return "out_of_bounds"

def shots_from_bounces(analysis, ball_track, near_track, far_track,
                       near_handed='left', far_handed='right', fps=25,
                       look_ahead=25, contact_thresh=120):
    in_bounces = [(f, coord) for f, coord, v in analysis if v.startswith('IN')]
    shots = []
    for bf, coord in in_bounces:
        side = 'far' if coord[1] < 1748 else 'near'
        track = far_track if side=='far' else near_track
        handed = far_handed if side=='far' else near_handed
        zone = zone_of(coord)   # where this bounce landed

        best = None
        for f in range(bf, min(bf+look_ahead, len(ball_track))):
            b = ball_track[f]; box = track[f]
            if b[0] is None or box is None: continue
            cx = (box[0]+box[2])/2; cy=(box[1]+box[3])/2
            d = np.hypot(b[0]-cx, b[1]-cy)
            if best is None or d < best[1]:
                best = (f, d, b[0], cx)

        if best is None:
            shots.append((bf, None, side, 'NO DATA', zone, round(bf/fps,1)))
            continue
        cf, d, bx, cx = best
        if d > contact_thresh:
            shots.append((bf, None, side, 'NO CONTACT (winner/unreturned)', zone, round(bf/fps,1)))
            continue
        screen = 'right' if bx > cx else 'left'
        body = screen if side=='near' else ('left' if screen=='right' else 'right')
        stroke = 'FOREHAND' if body==handed else 'BACKHAND'
        shots.append((bf, cf, side, stroke, zone, round(cf/fps,1)))
    return shots

shots = shots_from_bounces(analysis, ball_track_big, near_track, far_track,
                           near_handed='left', far_handed='right')

print(f"{'bnc':>4} {'cf':>4} | {'player':6} | {'stroke':30} | {'landed in zone':20} | time")
print("-"*85)
for bf, cf, side, stroke, zone, t in shots:
    cf_str = str(cf) if cf is not None else " - "
    print(f"{bf:>4} {cf_str:>4} | {side:6} | {stroke:30} | {zone:20} | {t}s")

In [ ]:
!pip install "scenedetect[opencv]" -q
print("scenedetect ready")

In [ ]:
from google.colab import files
uploaded = files.upload()   # pick your match video

In [ ]:
%cd /content
!git clone https://github.com/yastrebksv/TennisProject.git
!pip install catboost -q
%cd /content/TennisProject
!gdown 1XEYZ4myUN7QT-NeBYJI0xteLsvs-ZAOl -O model_ball.pt
!gdown 1f-Co64ehgq4uddcQm1aFBDtbnyZhQvgG -O model_court.pt
!gdown 1Eo5HDnAQE8y_FbOftKZ8pjiojwuy2BmJ -O model_bounce.cbm
!find /content/TennisProject -name "*.py" -exec sed -i 's/np\.Inf/np.inf/g' {} +
from google.colab import drive
drive.mount('/content/drive')
print("REBUILD COMPLETE")

In [ ]:
import shutil, os
os.makedirs('/content/drive/MyDrive/tennis_weights', exist_ok=True)
for fn in ['model_ball.pt','model_court.pt','model_bounce.cbm']:
    shutil.copy(f'/content/TennisProject/{fn}', f'/content/drive/MyDrive/tennis_weights/{fn}')
print("Weights permanent in Drive")

In [ ]:
import os, cv2, json, csv, sys
import numpy as np
import torch, torchvision
from scipy.spatial import distance
sys.path.append('/content/TennisProject')
from court_reference import CourtReference

# ---------- court geometry ----------
X_LDBL, X_LSGL, X_CTR, X_RSGL, X_RDBL = 286, 423, 832, 1242, 1379
Y_FAR_BASE, Y_FAR_SVC, Y_NET, Y_NEAR_SVC, Y_NEAR_BASE = 561, 1110, 1748, 2386, 2935
COURT_X, COURT_Y, MARGIN = (286, 1379), (561, 2935), 30

ZONES = {
    "far_alley_left":      (X_LDBL, X_LSGL, Y_FAR_BASE, Y_NET),
    "far_alley_right":     (X_RSGL, X_RDBL, Y_FAR_BASE, Y_NET),
    "near_alley_left":     (X_LDBL, X_LSGL, Y_NET, Y_NEAR_BASE),
    "near_alley_right":    (X_RSGL, X_RDBL, Y_NET, Y_NEAR_BASE),
    "far_service_left":    (X_LSGL, X_CTR,  Y_FAR_SVC, Y_NET),
    "far_service_right":   (X_CTR,  X_RSGL, Y_FAR_SVC, Y_NET),
    "far_backcourt_left":  (X_LSGL, X_CTR,  Y_FAR_BASE, Y_FAR_SVC),
    "far_backcourt_right": (X_CTR,  X_RSGL, Y_FAR_BASE, Y_FAR_SVC),
    "near_service_left":   (X_LSGL, X_CTR,  Y_NET, Y_NEAR_SVC),
    "near_service_right":  (X_CTR,  X_RSGL, Y_NET, Y_NEAR_SVC),
    "near_backcourt_left": (X_LSGL, X_CTR,  Y_NEAR_SVC, Y_NEAR_BASE),
    "near_backcourt_right":(X_CTR,  X_RSGL, Y_NEAR_SVC, Y_NEAR_BASE),
}

def zone_of(coord):
    if coord is None: return "unknown"
    cx, cy = coord
    for name, (x0, x1, y0, y1) in ZONES.items():
        if x0 <= cx <= x1 and y0 <= cy <= y1:
            return name
    return "out_of_bounds"

# ---------- ball track helpers ----------
def interpolate_gaps(ball_track, max_gap=60):
    track = list(ball_track)
    det = [i for i in range(len(track)) if track[i][0] is not None]
    for k in range(len(det) - 1):
        i0, i1 = det[k], det[k+1]
        gap = i1 - i0 - 1
        if 1 <= gap <= max_gap:
            x0, y0 = track[i0]
            x1, y1 = track[i1]
            for j in range(1, gap + 1):
                t = j / (gap + 1)
                track[i0 + j] = (x0 + (x1 - x0) * t, y0 + (y1 - y0) * t)
    return track

def to_court(f, ball_track, H):
    pos, mat = ball_track[f], H[f]
    if pos[0] is None or mat is None: return None
    pt = np.array(pos, dtype=np.float32).reshape(1, 1, 2)
    o = cv2.perspectiveTransform(pt, mat)
    return (float(o[0, 0, 0]), float(o[0, 0, 1]))

def on_court(p, margin=MARGIN):
    if p is None: return None
    x, y = p
    return (COURT_X[0]-margin <= x <= COURT_X[1]+margin and
            COURT_Y[0]-margin <= y <= COURT_Y[1]+margin)

def dir_change(f, ball_track, w=3):
    ys = [ball_track[i][1] if 0 <= i < len(ball_track) and ball_track[i][0] is not None else None
          for i in range(f-w, f+w+1)]
    before = [y for y in ys[:w] if y is not None]
    after  = [y for y in ys[w+1:] if y is not None]
    if not before or not after or ys[w] is None: return None
    return ys[w] >= max(before) - 2 and ys[w] >= max(after) - 2

# ---------- bounce classification + side-alternation collapse ----------
def classify_bounces(bounces, ball_track, H, cluster_window=8):
    classified = []
    for f in sorted(bounces):
        p = to_court(f, ball_track, H)
        classified.append((f, p, on_court(p), dir_change(f, ball_track)))

    results, i, n = [], 0, len(classified)
    while i < n:
        f, p, oc, dc = classified[i]
        if oc is True:
            j = i
            while (j+1 < n and classified[j+1][2] is True
                   and classified[j+1][0] - classified[j][0] <= cluster_window):
                j += 1
            results.append((classified[j][0], classified[j][1], 'IN'))
            i = j + 1
        elif oc is False:
            had_real = any(r[2] == 'IN' for r in results)
            results.append((f, p, 'FALSE' if (had_real and dc) else 'OUT'))
            i += 1
        else:
            i += 1

    collapsed, run, prev = [], [], None
    for e in results:
        if e[2] != 'IN':
            if run: collapsed.append(run[-1]); run = []
            prev = None; collapsed.append(e); continue
        s = 'far' if e[1][1] < Y_NET else 'near'
        # only merge into the current run if SAME side AND close in time (true duplicate);
        # a same-side bounce far apart in time is a separate real bounce
        if s == prev and run and (e[0] - run[-1][0]) <= cluster_window:
            run.append(e)
        else:
            if run: collapsed.append(run[-1])
            run = [e]; prev = s
    if run: collapsed.append(run[-1])
    return collapsed

# ---------- player tracking by baseline distance ----------
def track_players(frames, H, kps_court, det_model, device):
    def detect(img, min_score=0.85):
        t = torch.from_numpy(img.transpose(2,0,1)/255).unsqueeze(0).float().to(device)
        with torch.no_grad():
            pred = det_model(t)[0]
        return [b.cpu().numpy() for b, l, s in
                zip(pred['boxes'], pred['labels'], pred['scores'])
                if l.item() == 1 and s.item() > min_score]

    def d2line(p, a, b):
        a, b, p = np.array(a, float), np.array(b, float), np.array(p, float)
        if np.allclose(a, b): return np.linalg.norm(p - a)
        return abs(np.cross(b - a, p - a)) / np.linalg.norm(b - a)

    far_t  = [None] * len(frames)
    near_t = [None] * len(frames)
    for f in range(len(frames)):
        kc = kps_court[f]
        if kc is None: continue
        def pt(i): return np.array(kc[i]).flatten()[:2]
        fa, fb = pt(0), pt(1)     # far baseline endpoints
        na, nb = pt(2), pt(3)     # near baseline endpoints
        ppl = detect(frames[f])
        if not ppl: continue
        fbest, nbest = 1e9, 1e9
        for b in ppl:
            foot = ((b[0]+b[2])/2, b[3])
            df, dn = d2line(foot, fa, fb), d2line(foot, na, nb)
            if df < fbest: fbest, far_t[f]  = df, b
            if dn < nbest: nbest, near_t[f] = dn, b
    return far_t, near_t

# ---------- MAIN: one clip -> shot rows ----------
def extract_shots(clip_path, rally_id, models, device,
                  near_handed='left', far_handed='right',
                  look_ahead=25, contact_thresh=120):
    ball_detector, court_detector, bounce_detector, det_model = models

    cap = cv2.VideoCapture(clip_path)
    frames = []
    while True:
        ret, f = cap.read()
        if not ret: break
        frames.append(cv2.resize(f, (1280, 720)))
    cap.release()
    if len(frames) < 30:
        return []

    ball_track = interpolate_gaps(ball_detector.infer_model(frames), max_gap=60)
    H, kps_court = court_detector.infer_model(frames)
    bounces = bounce_detector.predict([b[0] for b in ball_track],
                                      [b[1] for b in ball_track])
    classified = classify_bounces(bounces, ball_track, H)
    far_t, near_t = track_players(frames, H, kps_court, det_model, device)

    rows, sn = [], 0
    for bf, coord, verdict in classified:
        if verdict != 'IN': continue
        side   = 'far' if coord[1] < Y_NET else 'near'
        track  = far_t if side == 'far' else near_t
        handed = far_handed if side == 'far' else near_handed
        zone   = zone_of(coord)

        best = None
        for f in range(bf, min(bf + look_ahead, len(ball_track))):
            b, box = ball_track[f], track[f]
            if b[0] is None or box is None: continue
            cx, cy = (box[0]+box[2])/2, (box[1]+box[3])/2
            d = np.hypot(b[0]-cx, b[1]-cy)
            if best is None or d < best[1]:
                best = (f, d, b[0], cx)

        sn += 1
        if best is None or best[1] > contact_thresh:
            rows.append(dict(rally_id=rally_id, shot_num=sn, player=side,
                             stroke='NO_CONTACT', landing_zone=zone,
                             bounce_frame=bf, in_out='IN', is_winner=True))
            continue

        cf, d, bx, cx = best
        screen = 'right' if bx > cx else 'left'
        body   = screen if side == 'near' else ('left' if screen == 'right' else 'right')
        rows.append(dict(rally_id=rally_id, shot_num=sn, player=side,
                         stroke='FOREHAND' if body == handed else 'BACKHAND',
                         landing_zone=zone, bounce_frame=bf,
                         in_out='IN', is_winner=False))

    for bf, coord, verdict in classified:
        if verdict == 'OUT':
            sn += 1
            rows.append(dict(rally_id=rally_id, shot_num=sn, player='unknown',
                             stroke='OUT_BALL', landing_zone=zone_of(coord),
                             bounce_frame=bf, in_out='OUT', is_winner=False))
    return rows

print("Extractor functions defined")

In [ ]:
import torch, torchvision, os, json, sys, time, csv
sys.path.append('/content/TennisProject')
from ball_detector import BallDetector
from court_detection_net import CourtDetectorNet
from bounce_detector import BounceDetector

device = 'cuda' if torch.cuda.is_available() else 'cpu'
models = (
    BallDetector('/content/TennisProject/model_ball.pt', device),
    CourtDetectorNet('/content/TennisProject/model_court.pt', device),
    BounceDetector('/content/TennisProject/model_bounce.cbm'),
    torchvision.models.detection.fasterrcnn_resnet50_fpn(weights='DEFAULT').to(device).eval(),
)
print("Models loaded on", device)

In [ ]:
clip_dir = '/content/drive/MyDrive/rallies_match1'
with open('/content/drive/MyDrive/real_rallies_match1.json') as f:
    real_rallies = json.load(f)
print(f"{len(real_rallies)} rally clips")

for clip in real_rallies[:3]:
    t0 = time.time()
    rows = extract_shots(os.path.join(clip_dir, clip), clip, models, device,
                         near_handed='left', far_handed='right')
    print(f"\n{clip}: {len(rows)} shots in {time.time()-t0:.0f}s")
    for r in rows: print("   ", r)

In [ ]:
import cv2, numpy as np, os

def render_annotated_rally(clip_path, rally_id, models, device,
                           near_handed='left', far_handed='right',
                           out_path=None):
    """Re-run extraction on a clip and draw everything it found onto the video."""
    ball_detector, court_detector, bounce_detector, det_model = models

    cap = cv2.VideoCapture(clip_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    frames = []
    while True:
        ret, f = cap.read()
        if not ret: break
        frames.append(cv2.resize(f, (1280,720)))
    cap.release()

    ball_track = interpolate_gaps(ball_detector.infer_model(frames), max_gap=60)
    H, kps_court = court_detector.infer_model(frames)
    bounces = bounce_detector.predict([b[0] for b in ball_track],[b[1] for b in ball_track])
    classified = classify_bounces(bounces, ball_track, H)
    far_t, near_t = track_players(frames, H, kps_court, det_model, device)

    # build the shot rows (same logic as extract_shots) so we can annotate them
    shots = []
    sn = 0
    for bf, coord, verdict in classified:
        if verdict != 'IN': continue
        side = 'far' if coord[1] < Y_NET else 'near'
        track = far_t if side=='far' else near_t
        handed = far_handed if side=='far' else near_handed
        best = None
        for f in range(bf, min(bf+25, len(ball_track))):
            b, box = ball_track[f], track[f]
            if b[0] is None or box is None: continue
            cx = (box[0]+box[2])/2; cy=(box[1]+box[3])/2
            d = np.hypot(b[0]-cx, b[1]-cy)
            if best is None or d < best[1]: best = (f,d,b[0],cx)
        sn += 1
        if best is None or best[1] > 120:
            shots.append((bf, None, side, 'WINNER/NO-CONTACT', zone_of(coord)))
        else:
            cf,d,bx,cx = best
            screen = 'right' if bx>cx else 'left'
            body = screen if side=='near' else ('left' if screen=='right' else 'right')
            shots.append((bf, cf, side, 'FOREHAND' if body==handed else 'BACKHAND', zone_of(coord)))

    bounce_map  = {bf:(side,stroke,zone) for bf,cf,side,stroke,zone in shots}
    contact_map = {cf:(side,stroke) for bf,cf,side,stroke,zone in shots if cf is not None}

    if out_path is None:
        out_path = f'/content/annotated_{rally_id}'
    out = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (1280,720))

    for f in range(len(frames)):
        img = frames[f].copy()
        # player boxes
        for box,color,lbl in [(far_t[f],(255,0,0),'FAR'), (near_t[f],(0,0,255),'NEAR')]:
            if box is not None:
                x1,y1,x2,y2 = [int(v) for v in box]
                cv2.rectangle(img,(x1,y1),(x2,y2),color,2)
                cv2.putText(img,lbl,(x1,y1-6),cv2.FONT_HERSHEY_SIMPLEX,0.5,color,2)
        # ball trail
        for j in range(6):
            if f-j >= 0 and ball_track[f-j][0] is not None:
                cv2.circle(img,(int(ball_track[f-j][0]),int(ball_track[f-j][1])),
                           max(1,4-j//2),(0,255,255),-1)
        # BOUNCE marker
        if f in bounce_map:
            side,stroke,zone = bounce_map[f]
            b = ball_track[f]
            if b[0] is not None:
                cv2.circle(img,(int(b[0]),int(b[1])),14,(0,255,0),3)
            cv2.putText(img,f"BOUNCE ({side}) -> {zone}",(20,660),
                        cv2.FONT_HERSHEY_SIMPLEX,0.8,(0,255,0),2)
        # CONTACT / stroke marker
        if f in contact_map:
            side,stroke = contact_map[f]
            cv2.putText(img,f"{side.upper()} {stroke}",(20,700),
                        cv2.FONT_HERSHEY_SIMPLEX,0.9,(0,140,255),3)
        cv2.putText(img,f"F{f}",(20,35),cv2.FONT_HERSHEY_SIMPLEX,0.8,(255,255,255),2)
        out.write(img)
    out.release()
    print("Wrote", out_path)
    return shots

# render one of the clips you already extracted
clip_dir = '/content/drive/MyDrive/rallies_match1'
shots = render_annotated_rally(os.path.join(clip_dir,'rally_006.mp4'),
                               'rally_006.mp4', models, device)

In [ ]:
from google.colab import files
files.download('/content/annotated_rally_006.mp4')

In [ ]:
import cv2, os
clip = os.path.join('/content/drive/MyDrive/rallies_match1', 'rally_004.mp4')
cap = cv2.VideoCapture(clip)
n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"clip has {n} frames")
for label, idx in [('first', 0), ('middle', n//2), ('last', n-1)]:
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, f = cap.read()
    if ret:
        print(f"{label} frame shape: {f.shape}")
        crop = f[0:200, 0:600]   # top-left region
        cv2.imwrite(f'/content/scoreboard_{label}.png', crop)
cap.release()
print("saved crops")

In [ ]:
from google.colab import files
for lbl in ['first','middle','last']:
    files.download(f'/content/scoreboard_{lbl}.png')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import cv2, os
clip_dir = '/content/drive/MyDrive/rallies_match1'
# use a clip that's NOT the first (so the scoreboard is there)
cap = cv2.VideoCapture(os.path.join(clip_dir, 'rally_005.mp4'))
ret, frame = cap.read()
cap.release()
print("frame shape:", frame.shape)
cv2.imwrite('/content/full_frame.png', frame)

# try a few candidate crops of the top-left
for name, (y1,y2,x1,x2) in {
    'crop_a': (0, 150, 0, 500),
    'crop_b': (30, 130, 100, 450),
    'crop_c': (0, 200, 0, 700),
}.items():
    cv2.imwrite(f'/content/{name}.png', frame[y1:y2, x1:x2])
print("saved full_frame.png and 3 candidate crops")

In [ ]:
from google.colab import files
for f in ['full_frame.png','crop_a.png','crop_b.png','crop_c.png']:
    files.download(f'/content/{f}')

In [ ]:
!pip install easyocr -q
import easyocr, cv2, os
reader = easyocr.Reader(['en'], gpu=True)

clip_dir = '/content/drive/MyDrive/rallies_match1'

def read_scoreboard(clip_path, crop=(0,150,0,500), scale=3):
    y1,y2,x1,x2 = crop
    cap = cv2.VideoCapture(clip_path)
    ret, frame = cap.read()
    cap.release()
    if not ret: return None
    board = frame[y1:y2, x1:x2]
    board = cv2.resize(board, (board.shape[1]*scale, board.shape[0]*scale),
                       interpolation=cv2.INTER_CUBIC)
    results = reader.readtext(board)
    # return text with its x-position (helps us know column order)
    return [(text, conf, bbox[0][0]) for bbox, text, conf in results]

# test across clips from different parts of the match
for clip in ['rally_005.mp4','rally_050.mp4','rally_200.mp4','rally_400.mp4']:
    p = os.path.join(clip_dir, clip)
    if not os.path.exists(p): continue
    out = read_scoreboard(p)
    print(f"\n{clip}:")
    for text, conf, x in sorted(out, key=lambda t: t[2]):
        print(f"   x={x:5.0f}  '{text}'  (conf {conf:.2f})")

In [ ]:
def debug_scoreboard(clip_path, crop=(0,150,0,500), scale=3, n_tries=5):
    y1,y2,x1,x2 = crop
    cap = cv2.VideoCapture(clip_path)
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    idxs = [int(n*p) for p in np.linspace(0, 0.9, n_tries)]
    for i, idx in enumerate(idxs):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret: continue
        board = frame[y1:y2, x1:x2]
        board = cv2.resize(board, (board.shape[1]*scale, board.shape[0]*scale),
                           interpolation=cv2.INTER_CUBIC)
        results = reader.readtext(board)
        print(f"\n--- frame {idx} ({len(results)} items) ---")
        for b, t, c in sorted(results, key=lambda r: (r[0][0][1], r[0][0][0])):
            print(f"   y={b[0][1]:5.0f} x={b[0][0]:5.0f}  '{t}'  conf={c:.2f}")
    cap.release()

debug_scoreboard(os.path.join(clip_dir, 'rally_050.mp4'))

In [ ]:
import shutil, os
clip_dir = '/content/drive/MyDrive/rallies_match1'
clips = sorted([c for c in os.listdir(clip_dir) if c.endswith('.mp4')])

# zip the first 50 clips
os.makedirs('/content/batch', exist_ok=True)
for c in clips[:50]:
    shutil.copy(os.path.join(clip_dir, c), f'/content/batch/{c}')
shutil.make_archive('/content/clips_batch1', 'zip', '/content/batch')
print("zipped")

In [ ]:
import json, os, numpy as np

clip_dir = '/content/drive/MyDrive/rallies_match1'
with open('/content/drive/MyDrive/real_rallies_match1.json') as f:
    real_rallies = sorted(json.load(f))

def parse_score(ocr_out):
    """(sets_dj, sets_nd, games_dj, games_nd) from OCR output."""
    if not ocr_out: return None
    ys = [y for _,_,_,y in ocr_out]
    mid = (min(ys)+max(ys))/2
    top    = [(t,x) for t,c,x,y in ocr_out if y <  mid and t.isdigit()]
    bot    = [(t,x) for t,c,x,y in ocr_out if y >= mid and t.isdigit()]
    def col(row, lo, hi):
        c = [(t,x) for t,x in row if lo < x < hi]
        return int(c[0][0]) if c else None
    s_dj, s_nd = col(top,800,880), col(bot,800,880)
    g_dj, g_nd = col(top,880,950), col(bot,880,950)
    if None in (s_dj,s_nd,g_dj,g_nd): return None
    return (s_dj, s_nd, g_dj, g_nd)

def read_score(clip_path, crop=(0,150,0,500), scale=3, n_tries=6):
    y1,y2,x1,x2 = crop
    cap = cv2.VideoCapture(clip_path)
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    for idx in [int(n*p) for p in np.linspace(0,0.9,n_tries)]:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret: continue
        b = frame[y1:y2, x1:x2]
        b = cv2.resize(b,(b.shape[1]*scale,b.shape[0]*scale),interpolation=cv2.INTER_CUBIC)
        res = reader.readtext(b)
        p = parse_score([(t,c,bb[0][0],bb[0][1]) for bb,t,c in res])
        if p:
            cap.release(); return p
    cap.release(); return None

anchors = {}
for i, clip in enumerate(real_rallies):
    s = read_score(os.path.join(clip_dir, clip))
    if s: anchors[clip] = s
    if i % 25 == 0:
        print(f"  {i}/{len(real_rallies)}, {len(anchors)} anchors")

print(f"\n{len(anchors)} anchors from {len(real_rallies)} clips")
with open('/content/drive/MyDrive/score_anchors_match1.json','w') as f:
    json.dump(anchors, f)

In [ ]:
SET_GAMES = [10, 9, 13, 9, 18]   # from final score 6-4, 3-6, 7-6, 3-6, 10-8

def total_games(s_dj, s_nd, g_dj, g_nd):
    completed = s_dj + s_nd
    return sum(SET_GAMES[:completed]) + g_dj + g_nd

def nadal_near(score):
    N = total_games(*score)
    changeovers = (N + 1) // 2
    return changeovers % 2 == 0     # Nadal STARTS near

def handedness(score):
    return ('left','right') if nadal_near(score) else ('right','left')

# build a per-clip handedness map, interpolating for clips without anchors
clip_hand = {}
idx_of = {c:i for i,c in enumerate(real_rallies)}
anchor_idxs = sorted(idx_of[c] for c in anchors)

for clip in real_rallies:
    i = idx_of[clip]
    if clip in anchors:
        clip_hand[clip] = handedness(anchors[clip])
    else:
        # nearest anchor before and after
        before = [a for a in anchor_idxs if a < i]
        after  = [a for a in anchor_idxs if a > i]
        cands = []
        if before: cands.append(handedness(anchors[real_rallies[before[-1]]]))
        if after:  cands.append(handedness(anchors[real_rallies[after[0]]]))
        # if both neighbors agree, use it; else fall back to the nearest
        if len(cands)==2 and cands[0]==cands[1]:
            clip_hand[clip] = cands[0]
        elif cands:
            clip_hand[clip] = cands[0]
        else:
            clip_hand[clip] = ('left','right')   # default = start state

with open('/content/drive/MyDrive/handedness_match1.json','w') as f:
    json.dump({k:list(v) for k,v in clip_hand.items()}, f)
print("Handedness map built for", len(clip_hand), "clips")

In [ ]:
# print a few clips of each type to spot-check
lefty_near = [c for c,v in clip_hand.items() if v[0]=='left'][:5]
righty_near = [c for c,v in clip_hand.items() if v[0]=='right'][:5]
print("Map says NADAL (lefty) near:", lefty_near)
print("Map says DJOKOVIC (righty) near:", righty_near)

In [ ]:
import torch, torchvision, sys, os, json, csv, time
sys.path.append('/content/TennisProject')
from ball_detector import BallDetector
from court_detection_net import CourtDetectorNet
from bounce_detector import BounceDetector

device = 'cuda' if torch.cuda.is_available() else 'cpu'
models = (
    BallDetector('/content/TennisProject/model_ball.pt', device),
    CourtDetectorNet('/content/TennisProject/model_court.pt', device),
    BounceDetector('/content/TennisProject/model_bounce.cbm'),
    torchvision.models.detection.fasterrcnn_resnet50_fpn(weights='DEFAULT').to(device).eval(),
)
print("Models loaded")

In [ ]:
import os, json, csv, time, random

clip_dir = '/content/drive/MyDrive/rallies_match1'
with open('/content/drive/MyDrive/handedness_match1.json') as f:
    clip_hand = json.load(f)
with open('/content/drive/MyDrive/real_rallies_match1.json') as f:
    real_rallies = sorted(json.load(f))

random.seed(42)
subset = sorted(random.sample(real_rallies, 150))
print(f"Processing {len(subset)} clips")

out_csv = '/content/drive/MyDrive/shots_match1.csv'
fields = ['rally_id','shot_num','player','stroke','landing_zone',
          'bounce_frame','in_out','is_winner']

done = set()
if os.path.exists(out_csv):
    with open(out_csv) as f:
        done = {r['rally_id'] for r in csv.DictReader(f)}
    print(f"Resuming - {len(done)} already done")

t0 = time.time()
with open(out_csv, 'a', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fields)
    if not done: w.writeheader()
    for i, clip in enumerate(subset):
        if clip in done: continue
        near_h, far_h = clip_hand.get(clip, ['left','right'])
        try:
            rows = extract_shots(os.path.join(clip_dir, clip), clip, models, device,
                                 near_handed=near_h, far_handed=far_h)
            for r in rows: w.writerow(r)
            f.flush()
        except Exception as e:
            print(f"  {clip} FAILED: {e}")
        if i % 10 == 0 and i > 0:
            rate = (time.time() - t0) / i
            print(f"  {i}/{len(subset)} | ~{rate*(len(subset)-i)/60:.0f} min remaining")

print(f"\nDone in {(time.time()-t0)/60:.0f} min → {out_csv}")

In [ ]:
import csv
from collections import Counter

with open('/content/drive/MyDrive/shots_match1.csv') as f:
    rows = list(csv.DictReader(f))

clips = {r['rally_id'] for r in rows}
print(f"Clips: {len(clips)}")
print(f"Shot rows: {len(rows)}")
print(f"Avg shots/rally: {len(rows)/len(clips):.1f}\n")

print("Strokes:", Counter(r['stroke'] for r in rows))
print("Players:", Counter(r['player'] for r in rows))
print("Winners:", Counter(r['is_winner'] for r in rows))
print("\nTop zones:")
for z, n in Counter(r['landing_zone'] for r in rows).most_common(8):
    print(f"  {z}: {n}")

In [ ]:
import csv
from collections import defaultdict, Counter

with open('/content/drive/MyDrive/shots_match1.csv') as f:
    rows = list(csv.DictReader(f))

by_rally = defaultdict(list)
for r in rows:
    r['bounce_frame'] = int(r['bounce_frame'])
    r['shot_num'] = int(r['shot_num'])
    by_rally[r['rally_id']].append(r)

# how many out-balls per rally?
out_counts = Counter(sum(1 for r in v if r['stroke']=='OUT_BALL') for v in by_rally.values())
print("OUT_BALLs per rally:")
for n, c in sorted(out_counts.items()):
    print(f"  {n} out-balls: {c} rallies")

# show a few rallies with several out-balls
multi = [rid for rid, v in by_rally.items()
         if sum(1 for r in v if r['stroke']=='OUT_BALL') >= 2]
print(f"\n{len(multi)} rallies have 2+ out-balls. Showing 3:\n")
for rid in multi[:3]:
    print(f"--- {rid} ---")
    for r in sorted(by_rally[rid], key=lambda r: r['bounce_frame']):
        print(f"  frame {r['bounce_frame']:4d} | shot {r['shot_num']:2d} | "
              f"{r['player']:7s} | {r['stroke']:10s} | {r['landing_zone']:20s} | winner={r['is_winner']}")
    print()

In [ ]:
import csv
from collections import defaultdict, Counter

with open('/content/drive/MyDrive/shots_match1.csv') as f:
    rows = list(csv.DictReader(f))

by_rally = defaultdict(list)
for r in rows:
    r['bounce_frame'] = int(r['bounce_frame'])
    by_rally[r['rally_id']].append(r)

clean = []
for rid, rally in by_rally.items():
    rally.sort(key=lambda r: r['bounce_frame'])

    # an ending with real strokes AFTER it is a false ending -> drop it
    real_strokes = [r for r in rally if r['stroke'] in ('FOREHAND','BACKHAND')]
    last_stroke_frame = real_strokes[-1]['bounce_frame'] if real_strokes else -1

    seq = []
    for r in rally:
        is_ending = r['stroke'] in ('NO_CONTACT','OUT_BALL')
        if is_ending and r['bounce_frame'] < last_stroke_frame:
            continue
        seq.append(r)

    # truncate at the first surviving ending
    final = []
    for r in seq:
        final.append(r)
        if r['stroke'] in ('NO_CONTACT','OUT_BALL'):
            break

    for i, r in enumerate(final, 1):
        r['shot_num'] = i
        clean.append(r)

# ---- report ----
rallies = {r['rally_id'] for r in clean}
print(f"Clean rows: {len(clean)}  (was {len(rows)})")
print(f"Rallies: {len(rallies)}")
print(f"Avg shots/rally: {len(clean)/len(rallies):.1f}\n")
print("Strokes:", Counter(r['stroke'] for r in clean))
print("Players:", Counter(r['player'] for r in clean))

by_r = defaultdict(list)
for r in clean:
    by_r[r['rally_id']].append(r)
print("\nRally endings:", Counter(seq[-1]['stroke'] for seq in by_r.values()))

with open('/content/drive/MyDrive/shots_match1_clean.csv','w',newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(clean)
print("\nSaved shots_match1_clean.csv")

In [ ]:
from collections import Counter
import csv

print(f"Clean rows: {len(clean)} (was {len(rows)})")
rallies = {r['rally_id'] for r in clean}
print(f"Rallies: {len(rallies)}")
print(f"Avg shots/rally: {len(clean)/len(rallies):.1f}\n")
print("Strokes:", Counter(r['stroke'] for r in clean))
print("Players:", Counter(r['player'] for r in clean))

# how many rallies end in each way?
endings = Counter()
by_r = {}
for r in clean:
    by_r.setdefault(r['rally_id'], []).append(r)
for rid, seq in by_r.items():
    last = seq[-1]['stroke']
    endings[last] += 1
print("\nRally endings:", endings)

with open('/content/drive/MyDrive/shots_match1_clean.csv','w',newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(clean)
print("\nSaved shots_match1_clean.csv")

In [ ]:
import csv
from collections import defaultdict

with open('/content/drive/MyDrive/shots_match1_clean.csv') as f:
    rows = list(csv.DictReader(f))

by_rally = defaultdict(list)
for r in rows:
    by_rally[r['rally_id']].append(r)
for rid in by_rally:
    by_rally[rid].sort(key=lambda r: int(r['shot_num']))

# build (state -> machine's shot placement) training examples
examples = []
for rid, seq in by_rally.items():
    for i in range(len(seq)-1):
        cur, nxt = seq[i], seq[i+1]
        # the MACHINE is the near player -> we learn its shots
        if cur['player'] != 'near': continue
        if cur['stroke'] not in ('FOREHAND','BACKHAND'): continue
        # where the machine's shot landed = the next (far-side) bounce
        target = nxt['landing_zone']
        examples.append({
            'rally_id': rid,
            'shot_num': cur['shot_num'],
            'received_zone': cur['landing_zone'],   # where machine got the ball
            'machine_stroke': cur['stroke'],        # (for a real machine this is irrelevant)
            'target_zone': target,                  # WHERE THE MACHINE HIT IT -> the label
            'rally_ended': nxt['stroke'] in ('NO_CONTACT','OUT_BALL'),
        })

print(f"{len(examples)} machine-shot training examples")
for e in examples[:10]:
    print(f"  received {e['received_zone']:20s} -> hit to {e['target_zone']}")

In [ ]:
examples = [e for e in examples if e['target_zone'].startswith('far_')]
print(f"{len(examples)} examples (far-side targets only)\n")

zones = sorted({e['target_zone'] for e in examples})
print("Distribution:")
for z,n in Counter(e['target_zone'] for e in examples).most_common():
    print(f"  {z:22s} {n:3d} ({100*n/len(examples):.0f}%)")

# split by rally
rally_ids = sorted({e['rally_id'] for e in examples})
random.seed(0); random.shuffle(rally_ids)
cut = int(0.8*len(rally_ids))
train_ids = set(rally_ids[:cut]); test_ids = set(rally_ids[cut:])
train = [e for e in examples if e['rally_id'] in train_ids]
test  = [e for e in examples if e['rally_id'] in test_ids]
print(f"\ntrain {len(train)} / test {len(test)}\n")

most = Counter(e['target_zone'] for e in train).most_common(1)[0][0]
print(f"Random guess:        {100/len(zones):.1f}%")
print(f"Majority ({most[:16]}): {100*sum(1 for e in test if e['target_zone']==most)/len(test):.1f}%")

table = defaultdict(Counter)
for e in train: table[e['received_zone']][e['target_zone']] += 1
c = sum(1 for e in test if (table[e['received_zone']].most_common(1)[0][0]
                            if table.get(e['received_zone']) else most) == e['target_zone'])
print(f"Given received zone: {100*c/len(test):.1f}%")

table2 = defaultdict(Counter)
for e in train: table2[(e['received_zone'],e['stroke'])][e['target_zone']] += 1
c = sum(1 for e in test if (table2[(e['received_zone'],e['stroke'])].most_common(1)[0][0]
                            if table2.get((e['received_zone'],e['stroke'])) else most) == e['target_zone'])
print(f"+ stroke:            {100*c/len(test):.1f}%")

In [ ]:
"""
=============================================================================
TENNIS BALL MACHINE - RALLY EXTRACTION v2
=============================================================================
Extracts rich per-shot features for training a shot-placement policy.

WHY v2: the v1 zone-only features had ZERO predictive signal (33% baseline =
majority class). Tactical shot selection depends on WHERE THE PLAYERS ARE,
which v1 never captured. v2 fixes that.

PER-SHOT FEATURES CAPTURED
--------------------------
Identity / context
  rally_id, shot_num, player_side (near/far), player_name, handedness

Where the ball came to the hitter  (the STATE)
  recv_x, recv_y        precise court coords of the bounce they're answering
  recv_zone             12-zone label (kept - cheap)
  recv_depth            DEFENSE/CONTROL/PRESSURE/ATTACK/FINISH  (tactical band)

Where everyone is standing at the moment of contact  (the KEY MISSING PIECE)
  hitter_x, hitter_y    hitter's court position
  opp_x, opp_y          opponent's court position
  opp_open_side         which side of the court is open ('left'/'right'/'center')
  opp_displacement      how far the opponent is from their ideal center position

The shot they played  (the ACTION / LABEL)
  stroke                FOREHAND / BACKHAND
  tgt_x, tgt_y          precise coords where THIS shot landed  (= next bounce)
  tgt_zone              12-zone label of the target
  risk                  0..1 geometric risk of that placement (edges = risky)
  to_open_court         did they hit into the open side? (bool)

Outcome
  is_winner, rally_ended, contact_frame, bounce_frame
=============================================================================
"""

import os, cv2, csv, json, time
import numpy as np
import torch


# =============================================================================
# COURT GEOMETRY  (reference-court coordinate space)
# =============================================================================
X_LDBL, X_LSGL, X_CTR, X_RSGL, X_RDBL = 286, 423, 832, 1242, 1379
Y_FAR_BASE, Y_FAR_SVC, Y_NET, Y_NEAR_SVC, Y_NEAR_BASE = 561, 1110, 1748, 2386, 2935

COURT_X, COURT_Y, MARGIN = (286, 1379), (561, 2935), 30

# singles court half-dimensions (used for risk normalisation)
SINGLES_HALF_W = (X_RSGL - X_LSGL) / 2          # ~410
HALF_LEN       = (Y_NET - Y_FAR_BASE) / 2       # ~594

ZONES = {
    "far_alley_left":      (X_LDBL, X_LSGL, Y_FAR_BASE, Y_NET),
    "far_alley_right":     (X_RSGL, X_RDBL, Y_FAR_BASE, Y_NET),
    "near_alley_left":     (X_LDBL, X_LSGL, Y_NET, Y_NEAR_BASE),
    "near_alley_right":    (X_RSGL, X_RDBL, Y_NET, Y_NEAR_BASE),
    "far_service_left":    (X_LSGL, X_CTR,  Y_FAR_SVC, Y_NET),
    "far_service_right":   (X_CTR,  X_RSGL, Y_FAR_SVC, Y_NET),
    "far_backcourt_left":  (X_LSGL, X_CTR,  Y_FAR_BASE, Y_FAR_SVC),
    "far_backcourt_right": (X_CTR,  X_RSGL, Y_FAR_BASE, Y_FAR_SVC),
    "near_service_left":   (X_LSGL, X_CTR,  Y_NET, Y_NEAR_SVC),
    "near_service_right":  (X_CTR,  X_RSGL, Y_NET, Y_NEAR_SVC),
    "near_backcourt_left": (X_LSGL, X_CTR,  Y_NEAR_SVC, Y_NEAR_BASE),
    "near_backcourt_right":(X_CTR,  X_RSGL, Y_NEAR_SVC, Y_NEAR_BASE),
}


def zone_of(coord):
    if coord is None: return "unknown"
    cx, cy = coord
    for name, (x0, x1, y0, y1) in ZONES.items():
        if x0 <= cx <= x1 and y0 <= cy <= y1:
            return name
    return "out_of_bounds"


def depth_band(y, side):
    """
    Tactical depth band (the Defense -> Finish scale).
    A ball landing DEEP in your court pins you back (DEFENSE).
    A ball landing SHORT lets you attack (FINISH).
    """
    if side == 'far':
        # far court: y 561 (baseline) .. 1748 (net).  small y = deep
        depth = (y - Y_FAR_BASE) / (Y_NET - Y_FAR_BASE)
    else:
        # near court: y 2935 (baseline) .. 1748 (net).  large y = deep
        depth = (Y_NEAR_BASE - y) / (Y_NEAR_BASE - Y_NET)
    depth = max(0.0, min(1.0, depth))
    if depth < 0.20: return 'DEFENSE'     # pinned to the baseline
    if depth < 0.40: return 'CONTROL'
    if depth < 0.60: return 'PRESSURE'
    if depth < 0.80: return 'ATTACK'
    return 'FINISH'                        # short ball - put it away


def shot_risk(x, y, side):
    """
    Geometric risk of a placement: 0 = dead centre (safe), 1 = painting a line.
    Risk is driven by the THINNEST margin (sideline or baseline), because that
    is the margin that will burn you.
    """
    d_side = min(x - X_LSGL, X_RSGL - x)          # margin to nearest sideline
    if side == 'far':
        d_deep, d_short = y - Y_FAR_BASE, Y_NET - y
    else:
        d_deep, d_short = Y_NEAR_BASE - y, y - Y_NET
    d_len = min(d_deep, d_short)                  # margin to baseline or net

    side_risk  = 1 - (d_side / SINGLES_HALF_W)
    depth_risk = 1 - (d_len  / HALF_LEN)
    return round(float(max(side_risk, depth_risk)), 3)


def open_court(opp_x):
    """Which side is open, given where the opponent is standing."""
    if opp_x is None: return 'unknown'
    if opp_x < X_CTR - 120: return 'right'    # they're left -> right is open
    if opp_x > X_CTR + 120: return 'left'
    return 'center'


# =============================================================================
# BALL / BOUNCE HELPERS
# =============================================================================
def interpolate_gaps(ball_track, max_gap=60):
    track = list(ball_track)
    det = [i for i in range(len(track)) if track[i][0] is not None]
    for k in range(len(det) - 1):
        i0, i1 = det[k], det[k+1]
        gap = i1 - i0 - 1
        if 1 <= gap <= max_gap:
            x0, y0 = track[i0]; x1, y1 = track[i1]
            for j in range(1, gap + 1):
                t = j / (gap + 1)
                track[i0 + j] = (x0 + (x1-x0)*t, y0 + (y1-y0)*t)
    return track


def to_court(f, ball_track, H):
    pos, mat = ball_track[f], H[f]
    if pos[0] is None or mat is None: return None
    pt = np.array(pos, dtype=np.float32).reshape(1, 1, 2)
    o = cv2.perspectiveTransform(pt, mat)
    return (float(o[0, 0, 0]), float(o[0, 0, 1]))


def player_to_court(box, H_frame):
    """Transform a player's FEET (bottom-centre of box) into court coords."""
    if box is None or H_frame is None: return (None, None)
    foot = np.array([[[(box[0] + box[2]) / 2, box[3]]]], dtype=np.float32)
    p = cv2.perspectiveTransform(foot, H_frame)[0, 0]
    return (round(float(p[0]), 1), round(float(p[1]), 1))


def on_court(p, margin=MARGIN):
    if p is None: return None
    x, y = p
    return (COURT_X[0]-margin <= x <= COURT_X[1]+margin and
            COURT_Y[0]-margin <= y <= COURT_Y[1]+margin)


def dir_change(f, ball_track, w=3):
    ys = [ball_track[i][1] if 0 <= i < len(ball_track) and ball_track[i][0] is not None else None
          for i in range(f-w, f+w+1)]
    before = [y for y in ys[:w] if y is not None]
    after  = [y for y in ys[w+1:] if y is not None]
    if not before or not after or ys[w] is None: return None
    return ys[w] >= max(before) - 2 and ys[w] >= max(after) - 2


def classify_bounces(bounces, ball_track, H, cluster_window=8):
    """IN/OUT/FALSE, then collapse consecutive same-side IN bounces (keep latest)."""
    classified = []
    for f in sorted(bounces):
        p = to_court(f, ball_track, H)
        classified.append((f, p, on_court(p), dir_change(f, ball_track)))

    results, i, n = [], 0, len(classified)
    while i < n:
        f, p, oc, dc = classified[i]
        if oc is True:
            j = i
            while (j+1 < n and classified[j+1][2] is True
                   and classified[j+1][0] - classified[j][0] <= cluster_window):
                j += 1
            results.append((classified[j][0], classified[j][1], 'IN'))
            i = j + 1
        elif oc is False:
            had = any(r[2] == 'IN' for r in results)
            results.append((f, p, 'FALSE' if (had and dc) else 'OUT'))
            i += 1
        else:
            i += 1

    collapsed, run, prev = [], [], None
    for e in results:
        if e[2] != 'IN':
            if run: collapsed.append(run[-1]); run = []
            prev = None
            collapsed.append(e)
            continue
        s = 'far' if e[1][1] < Y_NET else 'near'
        if s == prev:
            run.append(e)
        else:
            if run: collapsed.append(run[-1])
            run = [e]; prev = s
    if run: collapsed.append(run[-1])
    return collapsed


# =============================================================================
# PLAYER TRACKING  (SPARSE - only near bounce frames.  ~3x faster)
# =============================================================================
def track_players_sparse(frames, kps_court, det_model, device, bounce_frames, window=30):
    def detect(img, min_score=0.85):
        t = torch.from_numpy(img.transpose(2,0,1)/255).unsqueeze(0).float().to(device)
        with torch.no_grad():
            pred = det_model(t)[0]
        return [b.cpu().numpy() for b, l, s in
                zip(pred['boxes'], pred['labels'], pred['scores'])
                if l.item() == 1 and s.item() > min_score]

    def d2line(p, a, b):
        a, b, p = np.array(a, float), np.array(b, float), np.array(p, float)
        if np.allclose(a, b): return np.linalg.norm(p - a)
        return abs(float(np.cross(b - a, p - a))) / np.linalg.norm(b - a)

    needed = set()
    for bf in bounce_frames:
        needed.update(range(bf, min(bf + window, len(frames))))

    far_t  = [None] * len(frames)
    near_t = [None] * len(frames)
    for f in sorted(needed):
        kc = kps_court[f]
        if kc is None: continue
        def pt(i): return np.array(kc[i]).flatten()[:2]
        fa, fb, na, nb = pt(0), pt(1), pt(2), pt(3)
        ppl = detect(frames[f])
        if not ppl: continue
        fbest, nbest = 1e9, 1e9
        for b in ppl:
            foot = ((b[0] + b[2]) / 2, b[3])
            df, dn = d2line(foot, fa, fb), d2line(foot, na, nb)
            if df < fbest: fbest, far_t[f]  = df, b
            if dn < nbest: nbest, near_t[f] = dn, b
    return far_t, near_t


# =============================================================================
# MAIN EXTRACTOR
# =============================================================================
def extract_shots_v2(clip_path, rally_id, models, device,
                     near_handed='left', far_handed='right',
                     near_name='NADAL', far_name='DJOKOVIC',
                     look_ahead=25, contact_thresh=120):
    ball_detector, court_detector, bounce_detector, det_model = models

    cap = cv2.VideoCapture(clip_path)
    frames = []
    while True:
        ret, f = cap.read()
        if not ret: break
        frames.append(cv2.resize(f, (1280, 720)))
    cap.release()
    if len(frames) < 30:
        return []

    ball_track = interpolate_gaps(ball_detector.infer_model(frames), max_gap=60)
    H, kps_court = court_detector.infer_model(frames)
    bounces = bounce_detector.predict([b[0] for b in ball_track],
                                      [b[1] for b in ball_track])
    classified = classify_bounces(bounces, ball_track, H)

    in_bounces = [(bf, c) for bf, c, v in classified if v == 'IN']
    if len(in_bounces) < 2:
        return []

    far_t, near_t = track_players_sparse(
        frames, kps_court, det_model, device,
        [bf for bf, _ in in_bounces], window=look_ahead + 5)

    rows = []
    for i in range(len(in_bounces) - 1):
        bf,   recv  = in_bounces[i]        # the bounce the hitter is answering
        nbf,  tgt   = in_bounces[i + 1]    # where their shot LANDED
        side = 'far' if recv[1] < Y_NET else 'near'

        # the target must be on the OTHER side - otherwise a bounce was missed
        tgt_side = 'far' if tgt[1] < Y_NET else 'near'
        if tgt_side == side:
            continue

        track   = far_t  if side == 'far' else near_t
        opp_trk = near_t if side == 'far' else far_t
        handed  = far_handed if side == 'far' else near_handed
        name    = far_name   if side == 'far' else near_name

        # find the contact frame: ball closest to the hitter after the bounce
        best = None
        for f in range(bf, min(bf + look_ahead, len(ball_track))):
            b, box = ball_track[f], track[f]
            if b[0] is None or box is None: continue
            cx, cy = (box[0]+box[2])/2, (box[1]+box[3])/2
            d = np.hypot(b[0]-cx, b[1]-cy)
            if best is None or d < best[1]:
                best = (f, d, b[0], cx)

        if best is None or best[1] > contact_thresh:
            rows.append(dict(
                rally_id=rally_id, shot_num=i+1, player_side=side,
                player_name=name, handedness=handed,
                recv_x=round(recv[0],1), recv_y=round(recv[1],1),
                recv_zone=zone_of(recv), recv_depth=depth_band(recv[1], side),
                hitter_x=None, hitter_y=None, opp_x=None, opp_y=None,
                opp_open_side='unknown', opp_displacement=None,
                stroke='NO_CONTACT',
                tgt_x=None, tgt_y=None, tgt_zone=None,
                risk=None, to_open_court=None,
                is_winner=True, contact_frame=None, bounce_frame=bf))
            break                                    # a winner ENDS the rally

        cf, d, bx, cx = best

        # positions at the moment of contact
        hx, hy = player_to_court(track[cf],   H[cf])
        ox, oy = player_to_court(opp_trk[cf], H[cf])

        # stroke: which side of the body was the ball on?
        screen = 'right' if bx > cx else 'left'
        body   = screen if side == 'near' else ('left' if screen == 'right' else 'right')
        stroke = 'FOREHAND' if body == handed else 'BACKHAND'

        open_side = open_court(ox)
        disp = round(abs(ox - X_CTR), 1) if ox is not None else None
        tgt_left = tgt[0] < X_CTR
        to_open = (open_side == 'left' and tgt_left) or (open_side == 'right' and not tgt_left)

        rows.append(dict(
            rally_id=rally_id, shot_num=i+1, player_side=side,
            player_name=name, handedness=handed,
            recv_x=round(recv[0],1), recv_y=round(recv[1],1),
            recv_zone=zone_of(recv), recv_depth=depth_band(recv[1], side),
            hitter_x=hx, hitter_y=hy, opp_x=ox, opp_y=oy,
            opp_open_side=open_side, opp_displacement=disp,
            stroke=stroke,
            tgt_x=round(tgt[0],1), tgt_y=round(tgt[1],1),
            tgt_zone=zone_of(tgt),
            risk=shot_risk(tgt[0], tgt[1], tgt_side),
            to_open_court=to_open,
            is_winner=False, contact_frame=cf, bounce_frame=bf))

    return rows


FIELDS = ['rally_id','shot_num','player_side','player_name','handedness',
          'recv_x','recv_y','recv_zone','recv_depth',
          'hitter_x','hitter_y','opp_x','opp_y','opp_open_side','opp_displacement',
          'stroke','tgt_x','tgt_y','tgt_zone','risk','to_open_court',
          'is_winner','contact_frame','bounce_frame']

In [ ]:
import time, csv, json, random, os

clip_dir = '/content/drive/MyDrive/rallies_match1'
with open('/content/drive/MyDrive/handedness_match1.json') as f:
    clip_hand = json.load(f)
with open('/content/drive/MyDrive/real_rallies_match1.json') as f:
    real_rallies = sorted(json.load(f))

random.seed(42)
subset = sorted(random.sample(real_rallies, 150))

out_csv = '/content/drive/MyDrive/shots_v2_match1.csv'
done = set()
if os.path.exists(out_csv):
    with open(out_csv) as f:
        done = {r['rally_id'] for r in csv.DictReader(f)}

t0 = time.time()
with open(out_csv, 'a', newline='') as f:
    w = csv.DictWriter(f, fieldnames=FIELDS)
    if not done: w.writeheader()
    for i, clip in enumerate(subset):
        if clip in done: continue
        near_h, far_h = clip_hand.get(clip, ['left','right'])
        near_name = 'NADAL' if near_h == 'left' else 'DJOKOVIC'
        far_name  = 'NADAL' if far_h  == 'left' else 'DJOKOVIC'
        try:
            rows = extract_shots_v2(os.path.join(clip_dir, clip), clip, models, device,
                                    near_handed=near_h, far_handed=far_h,
                                    near_name=near_name, far_name=far_name)
            for r in rows: w.writerow(r)
            f.flush()
        except Exception as e:
            print(f"  {clip} FAILED: {e}")
        if i % 10 == 0 and i > 0:
            rate = (time.time()-t0)/i
            print(f"  {i}/{len(subset)} | ~{rate*(len(subset)-i)/60:.0f} min left")

print(f"Done in {(time.time()-t0)/60:.0f} min")

In [ ]:
%cd /content
!git clone https://github.com/yastrebksv/TennisProject.git
!pip install catboost -q
%cd /content/TennisProject
!gdown 1XEYZ4myUN7QT-NeBYJI0xteLsvs-ZAOl -O model_ball.pt
!gdown 1f-Co64ehgq4uddcQm1aFBDtbnyZhQvgG -O model_court.pt
!gdown 1Eo5HDnAQE8y_FbOftKZ8pjiojwuy2BmJ -O model_bounce.cbm
!find /content/TennisProject -name "*.py" -exec sed -i 's/np\.Inf/np.inf/g' {} +
from google.colab import drive
drive.mount('/content/drive')
print("REBUILD COMPLETE")

In [ ]:
import shutil, os
os.makedirs('/content/drive/MyDrive/tennis_weights', exist_ok=True)
for fn in ['model_ball.pt','model_court.pt','model_bounce.cbm']:
    src = f'/content/TennisProject/{fn}'
    if os.path.exists(src):
        shutil.copy(src, f'/content/drive/MyDrive/tennis_weights/{fn}')
        print(f"{fn} -> Drive")